<a href="https://colab.research.google.com/github/hoangchaulanbao/HocmayNC/blob/main/chatbot_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📘 PIPELINE HOÀN CHỈNH: CHATBOT PHÁP LUẬT DÂN SỰ VN
Mục tiêu: Xây RAG chatbot trả lời câu hỏi về Bộ luật Dân sự 2015 (689 Điều)
Stack: Qwen 2.5-3B + multilingual-e5-large + ChromaDB + Gradio

Môi trường: Google Colab + T4 GPU + Google Drive

# 🔧 BƯỚC 0: CHUẨN BỊ THỦ CÔNG (làm 1 lần)0.1. Bật GPU trên Colab

Menu Runtime → Change runtime type → chọn T4 GPU → Save
0.2. Chuẩn bị file BLDS 2015

Tải bản .docx (Word) của BLDS 2015 từ thuvienphapluat.vn hoặc vbpl.vn
Nếu file là .doc cũ → convert thành .docx (qua Google Docs hoặc cloudconvert.com)
Upload lên Google Drive: MyDrive/legal_chatbot_civil/corpus/blds.docx

# 📦 CELL 1: CÀI ĐẶT PACKAGES




In [22]:
# ============================================================
# CELL 1: CÀI ĐẶT PACKAGES
# ============================================================
# QUAN TRỌNG: KHÔNG cài torch để giữ nguyên torch GPU của Colab

!pip install -q --upgrade \
    "transformers>=4.44.0,<5.0.0" \
    "accelerate>=0.33.0" \
    "bitsandbytes>=0.43.0" \
    "peft>=0.12.0,<1.0.0" \
    "trl>=0.9.0,<1.0.0" \
    "datasets>=2.20.0,<4.0.0" \
    "sentence-transformers>=3.0.0,<5.0.0" \
    "chromadb>=0.5.0,<1.0.0" \
    "ragas>=0.1.10" \
    "gradio>=4.36.0,<5.0.0" \
    "pandas>=2.2.0,<3.0.0" \
    "python-docx" \
    einops tiktoken \
    underthesea beautifulsoup4 requests \
    matplotlib seaborn tqdm

# Fix opentelemetry để chromadb import được
!pip install -q --force-reinstall \
    "opentelemetry-api==1.38.0" \
    "opentelemetry-sdk==1.38.0" \
    "opentelemetry-proto==1.38.0" \
    "opentelemetry-exporter-otlp-proto-common==1.38.0" \
    "opentelemetry-exporter-otlp-proto-grpc==1.38.0" \
    "opentelemetry-exporter-otlp-proto-http==1.38.0"

print("\n✅ Cài đặt xong.")
print("⚠️ BẮT BUỘC: Runtime → Restart session, rồi chạy Cell 2.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 8.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 99.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.7/345.7 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.3/277.3 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 110.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# 🔍 CELL 2: VERIFY MÔI TRƯỜNG

In [1]:
# ============================================================
# CELL 2: VERIFY GPU & PACKAGES (chạy sau khi restart)
# ============================================================
import torch
import transformers, peft, trl, datasets
import sentence_transformers, chromadb, gradio
import pandas as pd
import bitsandbytes as bnb

print("=" * 60)
print("🔥 GPU & PyTorch")
print("=" * 60)
print(f"torch:        {torch.__version__}")
print(f"CUDA:         {torch.cuda.is_available()}")
print(f"GPU:          {torch.cuda.get_device_name(0)}")
print(f"VRAM:         {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\n" + "=" * 60)
print("📦 Packages")
print("=" * 60)
print(f"transformers:          {transformers.__version__}")
print(f"peft:                  {peft.__version__}")
print(f"trl:                   {trl.__version__}")
print(f"datasets:              {datasets.__version__}")
print(f"sentence-transformers: {sentence_transformers.__version__}")
print(f"chromadb:              {chromadb.__version__}")
print(f"gradio:                {gradio.__version__}")
print(f"pandas:                {pd.__version__}")
print(f"bitsandbytes:          {bnb.__version__}")
print("\n✅ Môi trường sẵn sàng!")

🔥 GPU & PyTorch
torch:        2.11.0+cu128
CUDA:         True
GPU:          Tesla T4
VRAM:         15.6 GB

📦 Packages
transformers:          4.57.6
peft:                  0.19.1
trl:                   0.29.1
datasets:              3.6.0
sentence-transformers: 4.1.0
chromadb:              0.6.3
gradio:                4.44.1
pandas:                2.3.3
bitsandbytes:          0.49.2

✅ Môi trường sẵn sàng!


# ⚙️ CELL 3: CONFIG

In [2]:
# ============================================================
# CELL 3: CẤU HÌNH HỆ THỐNG
# ============================================================
import os, json, re, time, torch, gc, hashlib
import requests
import numpy as np
import pandas as pd
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# --- Cảnh báo pháp lý ---
LEGAL_DISCLAIMER = (
    "⚠️ KHUYẾN CÁO: Thông tin chỉ mang tính THAM KHẢO dựa trên BLDS 2015. "
    "Không thay thế tư vấn pháp lý chuyên nghiệp. "
    "Hotline trợ giúp pháp lý miễn phí: 1900.6179"
)

SCOPE = "Bộ luật Dân sự 2015 (Luật số 91/2015/QH13, hiệu lực 01/01/2017)"

# Để rỗng - verify với chuyên gia rồi điền
AMENDED_ARTICLES = {}

CROSS_DOMAIN_KEYWORDS = {
    "Luật Hôn nhân và Gia đình 2014": [
        "ly hôn", "kết hôn", "cấp dưỡng", "quyền nuôi con", "tài sản vợ chồng",
        "hôn nhân", "gia đình", "vợ chồng", "con chung", "con riêng"
    ],
    "Luật Đất đai 2024": [
        "quyền sử dụng đất", "sổ đỏ", "thu hồi đất", "bồi thường đất",
        "chuyển nhượng đất", "giấy chứng nhận quyền sử dụng đất", "đất ở"
    ],
    "Luật Lao động 2019": [
        "hợp đồng lao động", "nghỉ việc", "sa thải", "lương", "bảo hiểm xã hội",
        "người lao động", "người sử dụng lao động"
    ],
    "Luật Thương mại 2005": [
        "mua bán hàng hóa", "đại lý thương mại", "nhượng quyền", "thương mại",
        "doanh nghiệp", "kinh doanh"
    ],
    "Luật Nhà ở 2023": [
        "mua bán nhà", "chung cư", "nhà ở xã hội", "sở hữu nhà", "thuê nhà"
    ],
    "Bộ luật Hình sự 2015": [
        "tội phạm", "hình sự", "trộm cắp", "lừa đảo", "giết người", "ma túy", "bạo lực"
    ],
}

@dataclass
class Config:
    # Models (tối ưu cho T4 15.8GB)
    llm_model_name: str = "Qwen/Qwen2.5-3B-Instruct"
    embedding_model_name: str = "intfloat/multilingual-e5-large"
    reranker_model_name: str = "BAAI/bge-reranker-base"

    # RAG
    chunk_size: int = 1500
    chunk_overlap: int = 200
    top_k_retrieval: int = 15
    top_k_rerank: int = 5

    # Confidence
    confidence_high: float = 0.55
    confidence_medium: float = 0.30
    ambiguity_chapter_threshold: int = 3

    # QLoRA
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    learning_rate: float = 2e-4
    num_epochs: int = 3
    batch_size: int = 1
    gradient_accumulation_steps: int = 16
    max_seq_length: int = 1024
    warmup_ratio: float = 0.1

    # Conversation
    max_history_turns: int = 5
    topic_change_threshold: float = 0.35

    # Drive paths
    drive_path: str = "/content/drive/MyDrive/legal_chatbot_civil"
    chroma_path: str = "/content/drive/MyDrive/legal_chatbot_civil/chroma_db"
    adapter_path: str = "/content/drive/MyDrive/legal_chatbot_civil/adapter"
    corpus_path: str = "/content/drive/MyDrive/legal_chatbot_civil/corpus"
    results_path: str = "/content/drive/MyDrive/legal_chatbot_civil/results"

config = Config()
print(f"✅ Config loaded | LLM: {config.llm_model_name}")

✅ Config loaded | LLM: Qwen/Qwen2.5-3B-Instruct


# 💾 CELL 4: MOUNT DRIVE & TẠO FOLDER

In [3]:
# ============================================================
# CELL 4: MOUNT GOOGLE DRIVE & TẠO FOLDER
# ============================================================
from google.colab import drive
import os

drive.mount('/content/drive')

for folder in [config.drive_path, config.chroma_path, config.adapter_path,
               config.corpus_path, config.results_path]:
    os.makedirs(folder, exist_ok=True)
    print(f"✅ {folder}")

# Verify file BLDS đã upload
print(f"\n📁 Files trong corpus:")
for f in os.listdir(config.corpus_path):
    size_kb = os.path.getsize(os.path.join(config.corpus_path, f)) / 1024
    print(f"   - {f} ({size_kb:.1f} KB)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ /content/drive/MyDrive/legal_chatbot_civil
✅ /content/drive/MyDrive/legal_chatbot_civil/chroma_db
✅ /content/drive/MyDrive/legal_chatbot_civil/adapter
✅ /content/drive/MyDrive/legal_chatbot_civil/corpus
✅ /content/drive/MyDrive/legal_chatbot_civil/results

📁 Files trong corpus:
   - blds.docx (135.5 KB)
   - blds.pdf (6519.1 KB)
   - blds_2015_parsed.json (744.5 KB)


Trước Cell 5: Upload file blds.docx vào folder corpus/ nếu chưa có

In [4]:
# ============================================================
# CELL 5: ĐỌC FILE WORD & PARSE THÀNH 689 ARTICLES
# ============================================================
# Có 2 nhánh:
#   - Nếu đã có cache JSON → load nhanh (1 giây)
#   - Nếu chưa có → đọc Word + parse + lưu cache

import docx
import glob
import json

cache_path = os.path.join(config.corpus_path, "blds_2015_parsed.json")

# === NHÁNH 1: Load cache nếu có ===
if os.path.exists(cache_path):
    with open(cache_path, "r", encoding="utf-8") as f:
        civil_code_articles = json.load(f)
    print(f"⚡ Loaded {len(civil_code_articles)} articles từ cache JSON")
    print(f"   (Bỏ qua bước parse Word vì đã có cache)")

# === NHÁNH 2: Parse Word từ đầu ===
else:
    # Tìm file .docx
    docx_files = [f for f in glob.glob(os.path.join(config.corpus_path, "*.docx"))
                  if not f.endswith('.doc')]

    if not docx_files:
        raise FileNotFoundError(f"❌ Không tìm thấy .docx trong {config.corpus_path}")

    docx_path = docx_files[0]
    print(f"📖 Đang đọc: {os.path.basename(docx_path)}")

    # Đọc Word
    doc = docx.Document(docx_path)
    paragraphs = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
    full_text = "\n".join(paragraphs)
    print(f"   Paragraphs: {len(paragraphs):,} | Ký tự: {len(full_text):,}")

    # Parse thành articles
    def parse_blds_to_articles(text):
        articles = []
        current_phan, current_phan_title = "", ""
        current_chuong, current_chuong_title = "", ""
        current_article = None
        pending_chuong_title = pending_phan_title = False

        for line in text.split('\n'):
            line = line.strip()
            if not line:
                continue

            # Phần
            if re.match(r'^Phần\s+thứ\s+\w+', line, re.IGNORECASE):
                if current_article and current_article["content"].strip():
                    articles.append(current_article)
                    current_article = None
                current_phan = line
                pending_phan_title = True
                continue
            if pending_phan_title:
                current_phan_title = line
                pending_phan_title = False
                continue

            # Chương
            if re.match(r'^Chương\s+[IVXLCDM]+\s*$', line, re.IGNORECASE):
                if current_article and current_article["content"].strip():
                    articles.append(current_article)
                    current_article = None
                current_chuong = line
                pending_chuong_title = True
                continue
            if pending_chuong_title:
                current_chuong_title = line
                pending_chuong_title = False
                continue

            # Điều
            dieu_match = re.match(r'^Điều\s+(\d+)\.\s*(.*)', line)
            if dieu_match:
                if current_article and current_article["content"].strip():
                    articles.append(current_article)
                current_article = {
                    "dieu": f"Điều {dieu_match.group(1)}",
                    "dieu_number": int(dieu_match.group(1)),
                    "title": dieu_match.group(2).strip(),
                    "content": "",
                    "chuong": current_chuong,
                    "chuong_title": current_chuong_title,
                    "phan": current_phan,
                    "phan_title": current_phan_title,
                    "source": SCOPE,
                    "effective_date": "01/01/2017",
                }
                continue

            if current_article is not None:
                current_article["content"] += line + "\n"

        if current_article and current_article["content"].strip():
            articles.append(current_article)
        return articles

    civil_code_articles = parse_blds_to_articles(full_text)

    # Save cache
    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(civil_code_articles, f, ensure_ascii=False, indent=2)
    print(f"💾 Đã lưu cache: {cache_path}")

# === THỐNG KÊ ===
print(f"\n{'=' * 60}")
print(f"📊 STATS: {len(civil_code_articles)} articles")
print(f"{'=' * 60}")

from collections import Counter
phan_count = Counter(a["phan"] for a in civil_code_articles)
for phan, count in phan_count.most_common():
    print(f"   {phan:30s}: {count:3d} Điều")

dieu_numbers = [a["dieu_number"] for a in civil_code_articles]
print(f"\n   Range: Điều {min(dieu_numbers)} → Điều {max(dieu_numbers)}")
print(f"   Số Chương unique: {len(set(a['chuong'] for a in civil_code_articles))}")

if len(civil_code_articles) >= 600:
    print(f"\n🎉 OK! Sẵn sàng qua bước embedding.")

⚡ Loaded 689 articles từ cache JSON
   (Bỏ qua bước parse Word vì đã có cache)

📊 STATS: 689 articles
   Phần thứ ba                   : 335 Điều
   Phần thứ nhất                 : 157 Điều
   Phần thứ hai                  : 116 Điều
   Phần thứ tư                   :  54 Điều
   Phần thứ năm                  :  25 Điều
   Phần thứ sáu                  :   2 Điều

   Range: Điều 1 → Điều 689
   Số Chương unique: 27

🎉 OK! Sẵn sàng qua bước embedding.


# 📖 CELL 5: ĐỌC & PARSE FILE WORD (LẦN ĐẦU)

In [5]:
# ============================================================
# CELL 5: ĐỌC FILE WORD & PARSE THÀNH 689 ARTICLES
# ============================================================
# Có 2 nhánh:
#   - Nếu đã có cache JSON → load nhanh (1 giây)
#   - Nếu chưa có → đọc Word + parse + lưu cache

import docx
import glob
import json

cache_path = os.path.join(config.corpus_path, "blds_2015_parsed.json")

# === NHÁNH 1: Load cache nếu có ===
if os.path.exists(cache_path):
    with open(cache_path, "r", encoding="utf-8") as f:
        civil_code_articles = json.load(f)
    print(f"⚡ Loaded {len(civil_code_articles)} articles từ cache JSON")
    print(f"   (Bỏ qua bước parse Word vì đã có cache)")

# === NHÁNH 2: Parse Word từ đầu ===
else:
    # Tìm file .docx
    docx_files = [f for f in glob.glob(os.path.join(config.corpus_path, "*.docx"))
                  if not f.endswith('.doc')]

    if not docx_files:
        raise FileNotFoundError(f"❌ Không tìm thấy .docx trong {config.corpus_path}")

    docx_path = docx_files[0]
    print(f"📖 Đang đọc: {os.path.basename(docx_path)}")

    # Đọc Word
    doc = docx.Document(docx_path)
    paragraphs = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
    full_text = "\n".join(paragraphs)
    print(f"   Paragraphs: {len(paragraphs):,} | Ký tự: {len(full_text):,}")

    # Parse thành articles
    def parse_blds_to_articles(text):
        articles = []
        current_phan, current_phan_title = "", ""
        current_chuong, current_chuong_title = "", ""
        current_article = None
        pending_chuong_title = pending_phan_title = False

        for line in text.split('\n'):
            line = line.strip()
            if not line:
                continue

            # Phần
            if re.match(r'^Phần\s+thứ\s+\w+', line, re.IGNORECASE):
                if current_article and current_article["content"].strip():
                    articles.append(current_article)
                    current_article = None
                current_phan = line
                pending_phan_title = True
                continue
            if pending_phan_title:
                current_phan_title = line
                pending_phan_title = False
                continue

            # Chương
            if re.match(r'^Chương\s+[IVXLCDM]+\s*$', line, re.IGNORECASE):
                if current_article and current_article["content"].strip():
                    articles.append(current_article)
                    current_article = None
                current_chuong = line
                pending_chuong_title = True
                continue
            if pending_chuong_title:
                current_chuong_title = line
                pending_chuong_title = False
                continue

            # Điều
            dieu_match = re.match(r'^Điều\s+(\d+)\.\s*(.*)', line)
            if dieu_match:
                if current_article and current_article["content"].strip():
                    articles.append(current_article)
                current_article = {
                    "dieu": f"Điều {dieu_match.group(1)}",
                    "dieu_number": int(dieu_match.group(1)),
                    "title": dieu_match.group(2).strip(),
                    "content": "",
                    "chuong": current_chuong,
                    "chuong_title": current_chuong_title,
                    "phan": current_phan,
                    "phan_title": current_phan_title,
                    "source": SCOPE,
                    "effective_date": "01/01/2017",
                }
                continue

            if current_article is not None:
                current_article["content"] += line + "\n"

        if current_article and current_article["content"].strip():
            articles.append(current_article)
        return articles

    civil_code_articles = parse_blds_to_articles(full_text)

    # Save cache
    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(civil_code_articles, f, ensure_ascii=False, indent=2)
    print(f"💾 Đã lưu cache: {cache_path}")

# === THỐNG KÊ ===
print(f"\n{'=' * 60}")
print(f"📊 STATS: {len(civil_code_articles)} articles")
print(f"{'=' * 60}")

from collections import Counter
phan_count = Counter(a["phan"] for a in civil_code_articles)
for phan, count in phan_count.most_common():
    print(f"   {phan:30s}: {count:3d} Điều")

dieu_numbers = [a["dieu_number"] for a in civil_code_articles]
print(f"\n   Range: Điều {min(dieu_numbers)} → Điều {max(dieu_numbers)}")
print(f"   Số Chương unique: {len(set(a['chuong'] for a in civil_code_articles))}")

if len(civil_code_articles) >= 600:
    print(f"\n🎉 OK! Sẵn sàng qua bước embedding.")

⚡ Loaded 689 articles từ cache JSON
   (Bỏ qua bước parse Word vì đã có cache)

📊 STATS: 689 articles
   Phần thứ ba                   : 335 Điều
   Phần thứ nhất                 : 157 Điều
   Phần thứ hai                  : 116 Điều
   Phần thứ tư                   :  54 Điều
   Phần thứ năm                  :  25 Điều
   Phần thứ sáu                  :   2 Điều

   Range: Điều 1 → Điều 689
   Số Chương unique: 27

🎉 OK! Sẵn sàng qua bước embedding.


# 🧠 CELL 5.1: LOAD EMBEDDING MODEL

In [6]:
# ============================================================
# CELL 5.1: LOAD EMBEDDING MODEL (intfloat/multilingual-e5-large)
# ============================================================
# Mục đích: Load model embedding → biến text thành vector 1024-dim
# VRAM: ~2.2GB (fp16)

from sentence_transformers import SentenceTransformer
import torch

print(f"📥 Loading embedding model: {config.embedding_model_name}")
print(f"   (Lần đầu sẽ download ~2.2GB, mất 2-5 phút)\n")

embedding_model = SentenceTransformer(
    config.embedding_model_name,
    device="cuda"
)
embedding_model.half()  # fp16 tiết kiệm VRAM

# Test
test_text = "Điều 1: Phạm vi điều chỉnh của Bộ luật Dân sự 2015"
test_embedding = embedding_model.encode(test_text, convert_to_numpy=True)

print(f"✅ Embedding model loaded!")
print(f"   Dimension: {test_embedding.shape[0]}")
print(f"   Device: {embedding_model.device}")
print(f"   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

📥 Loading embedding model: intfloat/multilingual-e5-large
   (Lần đầu sẽ download ~2.2GB, mất 2-5 phút)

✅ Embedding model loaded!
   Dimension: 1024
   Device: cuda:0
   VRAM used: 1.13 GB


# 🗄️ CELL 5.2: BUILD CHROMADB & EMBED 689 ARTICLES

In [7]:
# ============================================================
# CELL 5.2: BUILD CHROMADB & EMBED 689 ARTICLES
# ============================================================
import chromadb
from chromadb.config import Settings
import time

# Setup ChromaDB persistent client
print(f"📁 ChromaDB path: {config.chroma_path}")
os.makedirs(config.chroma_path, exist_ok=True)

chroma_client = chromadb.PersistentClient(
    path=config.chroma_path,
    settings=Settings(anonymized_telemetry=False)
)

COLLECTION_NAME = "blds_2015"

# Xóa collection cũ (nếu có) để build lại fresh
try:
    chroma_client.delete_collection(COLLECTION_NAME)
    print(f"🗑️  Xóa collection cũ")
except:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"description": "Bộ luật Dân sự 2015 - 689 Điều"}
)
print(f"✅ Tạo collection mới: {COLLECTION_NAME}\n")

# Chuẩn bị data
print(f"🔄 Embedding {len(civil_code_articles)} articles...")
documents, ids, metadatas = [], [], []

for art in civil_code_articles:
    # e5 model yêu cầu prefix "passage:" cho documents
    embed_text = (
        f"passage: {art['phan']} - {art['phan_title']}. "
        f"{art['chuong']} - {art['chuong_title']}. "
        f"{art['dieu']}: {art['title']}. "
        f"{art['content']}"
    )
    documents.append(embed_text)
    ids.append(f"dieu_{art['dieu_number']:03d}")
    metadatas.append({
        "dieu_number": art["dieu_number"],
        "dieu": art["dieu"],
        "title": art["title"],
        "chuong": art["chuong"],
        "chuong_title": art["chuong_title"],
        "phan": art["phan"],
        "phan_title": art["phan_title"],
        "content": art["content"],
        "source": art["source"],
    })

# Embed theo batch
BATCH_SIZE = 32
start = time.time()
all_embeddings = []

for i in range(0, len(documents), BATCH_SIZE):
    batch = documents[i:i+BATCH_SIZE]
    batch_emb = embedding_model.encode(
        batch,
        convert_to_numpy=True,
        show_progress_bar=False,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True
    )
    all_embeddings.extend(batch_emb.tolist())

    if (i + BATCH_SIZE) % 128 == 0 or i + BATCH_SIZE >= len(documents):
        done = min(i + BATCH_SIZE, len(documents))
        print(f"   {done}/{len(documents)} ({time.time()-start:.1f}s)")

print(f"\n✅ Embedded {len(all_embeddings)} vectors in {time.time()-start:.1f}s")

# Add vào ChromaDB
print(f"\n💾 Adding to ChromaDB...")
collection.add(
    ids=ids,
    embeddings=all_embeddings,
    documents=documents,
    metadatas=metadatas,
)

print(f"\n{'=' * 60}")
print(f"📊 CHROMADB STATS")
print(f"{'=' * 60}")
print(f"   Collection: {COLLECTION_NAME}")
print(f"   Total documents: {collection.count()}")
print(f"   Storage: {config.chroma_path}")

# Test query
test_query = "query: ly hôn và phân chia tài sản"
test_emb = embedding_model.encode(test_query, convert_to_numpy=True, normalize_embeddings=True)
test_results = collection.query(
    query_embeddings=[test_emb.tolist()],
    n_results=3
)

print(f"\n🔍 TEST QUERY: 'ly hôn và phân chia tài sản'")
print(f"   Top 3 results:")
for i, (id_, dist, meta) in enumerate(zip(
    test_results["ids"][0],
    test_results["distances"][0],
    test_results["metadatas"][0]
)):
    print(f"   {i+1}. [{id_}] {meta['dieu']}: {meta['title']}")
    print(f"      Distance: {dist:.4f} | {meta['chuong_title']}")

print(f"\n🎉 ChromaDB sẵn sàng cho RAG!")

📁 ChromaDB path: /content/drive/MyDrive/legal_chatbot_civil/chroma_db


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ Tạo collection mới: blds_2015

🔄 Embedding 689 articles...
   128/689 (2.0s)
   256/689 (4.0s)
   384/689 (5.3s)
   512/689 (6.9s)
   640/689 (8.5s)
   689/689 (9.3s)

✅ Embedded 689 vectors in 9.3s

💾 Adding to ChromaDB...


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



📊 CHROMADB STATS
   Collection: blds_2015
   Total documents: 689
   Storage: /content/drive/MyDrive/legal_chatbot_civil/chroma_db

🔍 TEST QUERY: 'ly hôn và phân chia tài sản'
   Top 3 results:
   1. [dieu_111] Điều 111: Vật chia được và vật không chia được
      Distance: 0.2672 | TÀI SẢN
   2. [dieu_657] Điều 657: Người phân chia di sản
      Distance: 0.2913 | THANH TOÁN VÀ PHÂN CHIA DI SẢN
   3. [dieu_213] Điều 213: Sở hữu chung của vợ chồng
      Distance: 0.2930 | QUYỀN SỞ HỮU

🎉 ChromaDB sẵn sàng cho RAG!


# Cell 6.1: Load Reranker Model

In [8]:
# ============================================================
# CELL 6.1: LOAD RERANKER MODEL (BAAI/bge-reranker-base)
# ============================================================
# Cross-encoder: nhận pair (query, document) → output relevance score
# VRAM: ~0.5GB (fp16)

from sentence_transformers import CrossEncoder
import torch

print(f"📥 Loading reranker: {config.reranker_model_name}")
print(f"   (Lần đầu sẽ download ~500MB, mất 1-2 phút)\n")

reranker_model = CrossEncoder(
    config.reranker_model_name,
    device="cuda",
    max_length=512  # Giới hạn độ dài input
)

# Test
test_pairs = [
    ("ly hôn và phân chia tài sản", "Điều 213: Sở hữu chung của vợ chồng"),
    ("ly hôn và phân chia tài sản", "Điều 100: Quốc tịch của pháp nhân"),
]
test_scores = reranker_model.predict(test_pairs)

print(f"✅ Reranker loaded!")
print(f"   VRAM total: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"\n🔍 Test scores (cao = liên quan hơn):")
print(f"   '{test_pairs[0][1]}' → {test_scores[0]:.3f}  (nên cao)")
print(f"   '{test_pairs[1][1]}' → {test_scores[1]:.3f}  (nên thấp)")

📥 Loading reranker: BAAI/bge-reranker-base
   (Lần đầu sẽ download ~500MB, mất 1-2 phút)



config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

✅ Reranker loaded!
   VRAM total: 2.25 GB

🔍 Test scores (cao = liên quan hơn):
   'Điều 213: Sở hữu chung của vợ chồng' → 0.088  (nên cao)
   'Điều 100: Quốc tịch của pháp nhân' → 0.001  (nên thấp)


# Cell 6.2: Retrieval Pipeline (Bi-encoder + Reranker)

In [9]:
# ============================================================
# CELL 6.2: RETRIEVAL PIPELINE (Bi-encoder + Reranker)
# ============================================================
# Hàm chính: retrieve(query) → list top-k articles liên quan nhất

import numpy as np
from typing import List, Dict
import time

def retrieve(
    query: str,
    top_k_retrieval: int = None,
    top_k_rerank: int = None,
    verbose: bool = False
) -> List[Dict]:
    """
    RAG retrieval với 2 tầng: embedding search + reranking.

    Args:
        query: Câu hỏi của user
        top_k_retrieval: Số documents lấy ở tầng 1 (default từ config)
        top_k_rerank: Số documents giữ lại sau rerank (default từ config)
        verbose: In log debug

    Returns:
        List of dicts, mỗi dict chứa:
            - dieu, dieu_number, title, content, chuong, phan
            - retrieval_score: cosine similarity (tầng 1)
            - rerank_score: cross-encoder score (tầng 2, càng cao càng liên quan)
            - rank: thứ tự sau rerank (1 = liên quan nhất)
    """
    if top_k_retrieval is None:
        top_k_retrieval = config.top_k_retrieval
    if top_k_rerank is None:
        top_k_rerank = config.top_k_rerank

    # ===== TẦNG 1: BI-ENCODER RETRIEVAL =====
    t0 = time.time()

    # e5 model yêu cầu prefix "query:" cho search query
    query_with_prefix = f"query: {query}"
    query_emb = embedding_model.encode(
        query_with_prefix,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Search trong ChromaDB
    results = collection.query(
        query_embeddings=[query_emb.tolist()],
        n_results=top_k_retrieval
    )

    t1 = time.time()

    candidates = []
    for i in range(len(results["ids"][0])):
        meta = results["metadatas"][0][i]
        # ChromaDB trả về distance (càng thấp càng gần) → convert sang similarity
        distance = results["distances"][0][i]
        similarity = 1 - distance  # Vì đã normalize, distance ∈ [0, 2]

        candidates.append({
            "dieu": meta["dieu"],
            "dieu_number": meta["dieu_number"],
            "title": meta["title"],
            "content": meta["content"],
            "chuong": meta["chuong"],
            "chuong_title": meta["chuong_title"],
            "phan": meta["phan"],
            "phan_title": meta["phan_title"],
            "retrieval_score": similarity,
        })

    if verbose:
        print(f"   🔍 Tầng 1 (embedding): {len(candidates)} candidates ({(t1-t0)*1000:.0f}ms)")

    # ===== TẦNG 2: CROSS-ENCODER RERANK =====
    t2 = time.time()

    # Build pairs (query, document) cho reranker
    pairs = []
    for c in candidates:
        # Combine title + content để reranker đánh giá đầy đủ
        doc_text = f"{c['dieu']}: {c['title']}. {c['content']}"
        # Cắt để vừa max_length của reranker
        doc_text = doc_text[:1500]
        pairs.append([query, doc_text])

    # Predict scores
    rerank_scores = reranker_model.predict(pairs, show_progress_bar=False)

    # Gán score vào candidates
    for c, score in zip(candidates, rerank_scores):
        c["rerank_score"] = float(score)

    # Sort theo rerank_score giảm dần
    candidates.sort(key=lambda x: x["rerank_score"], reverse=True)

    # Chỉ giữ top-k và gán rank
    top_results = candidates[:top_k_rerank]
    for i, c in enumerate(top_results):
        c["rank"] = i + 1

    t3 = time.time()

    if verbose:
        print(f"   🎯 Tầng 2 (rerank):    top {top_k_rerank} ({(t3-t2)*1000:.0f}ms)")
        print(f"   ⏱️  Tổng thời gian: {(t3-t0)*1000:.0f}ms")

    return top_results


# ===== TEST RETRIEVAL =====
print("=" * 70)
print("🧪 TEST RETRIEVAL PIPELINE")
print("=" * 70)

test_queries = [
    "Tôi muốn ly hôn, tài sản chung của vợ chồng được chia thế nào?",
    "Người dưới 18 tuổi có được ký hợp đồng mua bán nhà không?",
    "Thừa kế theo pháp luật được quy định ra sao?",
    "Hợp đồng vô hiệu là gì?",
    "Quyền sở hữu được xác lập khi nào?",
]

for query in test_queries:
    print(f"\n{'─' * 70}")
    print(f"❓ QUERY: {query}")
    print(f"{'─' * 70}")

    results = retrieve(query, verbose=True)

    print(f"\n📚 TOP {len(results)} ARTICLES:")
    for r in results:
        print(f"   #{r['rank']} [{r['dieu']}] {r['title']}")
        print(f"        Retrieval: {r['retrieval_score']:.3f} | Rerank: {r['rerank_score']:.3f}")
        print(f"        Chương: {r['chuong']} - {r['chuong_title']}")

print(f"\n{'=' * 70}")
print("🎉 Retrieval pipeline hoạt động!")

🧪 TEST RETRIEVAL PIPELINE

──────────────────────────────────────────────────────────────────────
❓ QUERY: Tôi muốn ly hôn, tài sản chung của vợ chồng được chia thế nào?
──────────────────────────────────────────────────────────────────────
   🔍 Tầng 1 (embedding): 15 candidates (72ms)
   🎯 Tầng 2 (rerank):    top 5 (310ms)
   ⏱️  Tổng thời gian: 382ms

📚 TOP 5 ARTICLES:
   #1 [Điều 655] Việc thừa kế trong trường hợp vợ, chồng đã chia tài sản chung; vợ, chồng đang xin ly hôn hoặc đã kết hôn với người khác
        Retrieval: 0.715 | Rerank: 0.956
        Chương: Chương XXIII - THỪA KẾ THEO PHÁP LUẬT
   #2 [Điều 213] Sở hữu chung của vợ chồng
        Retrieval: 0.726 | Rerank: 0.956
        Chương: Chương XIII - QUYỀN SỞ HỮU
   #3 [Điều 659] Phân chia di sản theo di chúc
        Retrieval: 0.673 | Rerank: 0.236
        Chương: Chương XXIV - THANH TOÁN VÀ PHÂN CHIA DI SẢN
   #4 [Điều 219] Chia tài sản thuộc sở hữu chung
        Retrieval: 0.695 | Rerank: 0.178
        Chương: Chương XIII 

# Cell 7: Load LLM Qwen 2.5-3B (4-bit quantization)
Giờ qua bước load LLM để generate câu trả lời. Mình dùng 4-bit quantization để tiết kiệm VRAM tối đa.
Cell 7: Load Qwen 2.5-3B (4-bit)

In [10]:
# ============================================================
# CELL 7: LOAD LLM QWEN 2.5-3B (4-BIT QUANTIZATION)
# ============================================================
# Mục đích: Load LLM để generate câu trả lời từ context retrieved
# Model: Qwen/Qwen2.5-3B-Instruct
#   - 3 tỷ tham số, hỗ trợ tiếng Việt tốt
#   - 4-bit quantization (NF4) → giảm VRAM từ ~6GB xuống ~2.5GB
#   - VRAM cần: ~2.5GB
# Tổng VRAM sau cell này: ~5-6 GB (embedding 2.2 + reranker 0.5 + LLM 2.5)
# Còn dư: ~10 GB → đủ cho generation context dài

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

print(f"📥 Loading LLM: {config.llm_model_name}")
print(f"   (Lần đầu sẽ download ~6GB, mất 3-7 phút)")
print(f"   VRAM hiện tại trước load: {torch.cuda.memory_allocated()/1e9:.2f} GB\n")

# === Cấu hình quantization 4-bit ===
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                          # Bật 4-bit
    bnb_4bit_quant_type="nf4",                  # NF4 (chất lượng tốt nhất)
    bnb_4bit_use_double_quant=True,             # Quantize cả quantization constants (tiết kiệm thêm ~0.4 bit/param)
    bnb_4bit_compute_dtype=torch.float16,       # Tính toán ở fp16
)

# === Load tokenizer ===
print("📝 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    config.llm_model_name,
    trust_remote_code=True,
)

# Set pad token (Qwen không có sẵn pad token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# === Load model ===
print("🧠 Loading model (4-bit)...")
llm_model = AutoModelForCausalLM.from_pretrained(
    config.llm_model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
llm_model.eval()  # Set eval mode (không train)

# === Stats ===
print(f"\n{'=' * 60}")
print(f"✅ LLM LOADED")
print(f"{'=' * 60}")
print(f"   Model: {config.llm_model_name}")
print(f"   Parameters: {sum(p.numel() for p in llm_model.parameters())/1e9:.2f}B")
print(f"   Quantization: 4-bit NF4")
print(f"   VRAM total: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"   VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1e9:.2f} GB")
print(f"   Vocab size: {len(tokenizer)}")

# === Test generation ===
print(f"\n🧪 TEST GENERATION...")
test_messages = [
    {"role": "system", "content": "Bạn là trợ lý pháp luật tiếng Việt."},
    {"role": "user", "content": "Chào bạn, bạn có thể giúp tôi tìm hiểu Bộ luật Dân sự 2015 không?"}
]

# Apply chat template
prompt = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.pad_token_id,
    )

response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(f"\n💬 Response:")
print(f"   {response}")
print(f"\n🎉 LLM ready!")

📥 Loading LLM: Qwen/Qwen2.5-3B-Instruct
   (Lần đầu sẽ download ~6GB, mất 3-7 phút)
   VRAM hiện tại trước load: 2.25 GB

📝 Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


✅ LLM LOADED
   Model: Qwen/Qwen2.5-3B-Instruct
   Parameters: 1.70B
   Quantization: 4-bit NF4
   VRAM total: 4.31 GB
   VRAM free: 11.32 GB
   Vocab size: 151665

🧪 TEST GENERATION...

💬 Response:
   Dĩ nhiên rồi! Bộ Luật Dân Sự 2015 được Quốc Hội thông qua và có hiệu lực từ ngày 01 tháng 01 năm 2017. Nó thay thế cho Bộ Luật Dân Sự 1999.

Bộ Luật này gồm nhiều quy định về các vấn đề như: quyền sở hữu bất động sản, quyền sở hữu trí tuệ, quyền lợi cá nhân dân sự, tranh chấp dân sự... 

Nếu bạn muốn tìm hiểu chi tiết hơn về từng điều khoản hoặc cụ thể về một vấn đề nào đó, tôi sẽ cố gắng hỗ trợ bạn theo hướng dẫn của bạn.

🎉 LLM ready!


# Cell 8: RAG Pipeline hoàn chỉnh (Retrieve + Generate)

In [11]:
# ============================================================
# CELL 8: RAG PIPELINE - RETRIEVE + GENERATE
# ============================================================
# Pipeline:
#   1. User hỏi câu hỏi
#   2. Retrieve top-5 Điều liên quan từ ChromaDB
#   3. Đánh giá confidence (cao/trung/thấp)
#   4. Build prompt với context + system instructions
#   5. LLM generate answer (grounded trong context)
#   6. Hậu xử lý: thêm disclaimer, trích dẫn điều luật

from typing import List, Dict, Tuple

# === SYSTEM PROMPT (Anti-hallucination) ===
SYSTEM_PROMPT = """Bạn là trợ lý pháp luật chuyên về Bộ luật Dân sự 2015 của Việt Nam.

NGUYÊN TẮC TRẢ LỜI (BẮT BUỘC TUÂN THỦ):
1. CHỈ trả lời dựa trên các Điều luật được cung cấp trong CONTEXT bên dưới.
2. KHÔNG bịa đặt thông tin. Nếu CONTEXT không đủ → trả lời "Tôi không tìm thấy thông tin chính xác trong BLDS 2015 cho câu hỏi này."
3. LUÔN trích dẫn số Điều cụ thể (vd: "Theo Điều 213 BLDS 2015...")
4. Trả lời NGẮN GỌN, RÕ RÀNG, đúng trọng tâm câu hỏi.
5. Dùng ngôn ngữ DỄ HIỂU cho người dân, tránh thuật ngữ pháp lý quá phức tạp.
6. Nếu câu hỏi liên quan luật khác (Hôn nhân, Đất đai, Lao động...) → nhắc user tham khảo luật đó.
7. KHÔNG đưa lời khuyên pháp lý cụ thể cho tình huống cá nhân — đó là việc của luật sư."""


def build_rag_prompt(query: str, retrieved_articles: List[Dict]) -> str:
    """Build prompt với context từ retrieved articles"""

    # Format context
    context_parts = []
    for art in retrieved_articles:
        context_parts.append(
            f"━━━ {art['dieu']}: {art['title']} ━━━\n"
            f"({art['chuong']} - {art['chuong_title']})\n"
            f"{art['content'].strip()}"
        )
    context = "\n\n".join(context_parts)

    # Build user message
    user_message = f"""CONTEXT - Các Điều luật liên quan trong Bộ luật Dân sự 2015:

{context}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

CÂU HỎI: {query}

Hãy trả lời câu hỏi trên dựa CHÍNH XÁC vào các Điều luật trong CONTEXT. Trích dẫn số Điều cụ thể."""

    return user_message


def evaluate_confidence(retrieved_articles: List[Dict]) -> Tuple[str, float, str]:
    """
    Đánh giá độ tin cậy dựa trên rerank scores.
    Returns: (level, top_score, emoji)
        level: "high" | "medium" | "low"
    """
    if not retrieved_articles:
        return "low", 0.0, "🔴"

    top_score = retrieved_articles[0]["rerank_score"]

    if top_score >= config.confidence_high:
        return "high", top_score, "🟢"
    elif top_score >= config.confidence_medium:
        return "medium", top_score, "🟡"
    else:
        return "low", top_score, "🔴"


def detect_cross_domain(query: str) -> List[str]:
    """Phát hiện câu hỏi có liên quan luật khác không"""
    query_lower = query.lower()
    related_laws = []

    for law, keywords in CROSS_DOMAIN_KEYWORDS.items():
        if any(kw in query_lower for kw in keywords):
            related_laws.append(law)

    return related_laws


def generate_answer(
    query: str,
    max_new_tokens: int = 512,
    temperature: float = 0.3,  # Thấp để giảm hallucination
    verbose: bool = True
) -> Dict:
    """
    Hàm chính: nhận câu hỏi → trả về câu trả lời + metadata.

    Returns dict:
        - answer: text trả lời
        - confidence: high/medium/low
        - retrieved_articles: list 5 điều luật
        - related_laws: list luật liên quan (nếu có)
        - took_ms: thời gian xử lý
    """
    import time
    t0 = time.time()

    # ===== BƯỚC 1: RETRIEVE =====
    if verbose:
        print(f"🔍 Retrieving...")
    retrieved = retrieve(query, verbose=False)

    # ===== BƯỚC 2: EVALUATE CONFIDENCE =====
    conf_level, conf_score, conf_emoji = evaluate_confidence(retrieved)
    if verbose:
        print(f"   Confidence: {conf_emoji} {conf_level.upper()} (score: {conf_score:.3f})")

    # ===== BƯỚC 3: CHECK LOW CONFIDENCE → REFUSE =====
    if conf_level == "low":
        return {
            "answer": (
                f"🔴 Tôi không tìm thấy thông tin đủ tin cậy trong Bộ luật Dân sự 2015 "
                f"để trả lời câu hỏi này (confidence: {conf_score:.3f}).\n\n"
                f"Bạn có thể:\n"
                f"• Diễn đạt lại câu hỏi cụ thể hơn\n"
                f"• Câu hỏi có thể thuộc luật khác (Hình sự, Hành chính, Lao động...)\n"
                f"• Liên hệ luật sư để được tư vấn chính xác\n\n"
                f"{LEGAL_DISCLAIMER}"
            ),
            "confidence": conf_level,
            "confidence_score": conf_score,
            "retrieved_articles": retrieved,
            "related_laws": [],
            "took_ms": int((time.time() - t0) * 1000),
        }

    # ===== BƯỚC 4: BUILD PROMPT =====
    if verbose:
        print(f"   Building prompt with {len(retrieved)} articles...")
    user_message = build_rag_prompt(query, retrieved)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # ===== BƯỚC 5: GENERATE =====
    if verbose:
        print(f"   Generating answer...")

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to("cuda")

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
        )

    answer_text = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    # ===== BƯỚC 6: POST-PROCESS =====
    # Detect cross-domain
    related_laws = detect_cross_domain(query)

    # Format final answer
    final_answer = f"{conf_emoji} **Độ tin cậy: {conf_level.upper()}** (score: {conf_score:.3f})\n\n"
    final_answer += answer_text + "\n\n"

    # Add điều luật references
    final_answer += "📚 **Tham khảo các Điều liên quan:**\n"
    for art in retrieved[:3]:
        final_answer += f"   • {art['dieu']}: {art['title']}\n"

    # Add cross-domain warning
    if related_laws:
        final_answer += f"\n⚠️ **Câu hỏi này có thể liên quan đến luật khác:**\n"
        for law in related_laws:
            final_answer += f"   • {law}\n"

    # Always add disclaimer
    final_answer += f"\n{LEGAL_DISCLAIMER}"

    took_ms = int((time.time() - t0) * 1000)
    if verbose:
        print(f"   ✅ Done in {took_ms}ms\n")

    return {
        "answer": final_answer,
        "confidence": conf_level,
        "confidence_score": conf_score,
        "retrieved_articles": retrieved,
        "related_laws": related_laws,
        "took_ms": took_ms,
    }


# ===== TEST RAG PIPELINE =====
print("=" * 70)
print("🧪 TEST RAG PIPELINE")
print("=" * 70)

test_questions = [
    "Tôi muốn ly hôn, tài sản chung của vợ chồng được chia thế nào?",
    "Người 16 tuổi có được ký hợp đồng mua bán nhà không?",
    "Hợp đồng vô hiệu là gì?",
]

for q in test_questions:
    print(f"\n{'═' * 70}")
    print(f"❓ CÂU HỎI: {q}")
    print(f"{'═' * 70}")

    result = generate_answer(q, verbose=True)

    print(f"💬 TRẢ LỜI ({result['took_ms']}ms):")
    print(f"{'─' * 70}")
    print(result["answer"])

🧪 TEST RAG PIPELINE

══════════════════════════════════════════════════════════════════════
❓ CÂU HỎI: Tôi muốn ly hôn, tài sản chung của vợ chồng được chia thế nào?
══════════════════════════════════════════════════════════════════════
🔍 Retrieving...
   Confidence: 🟢 HIGH (score: 0.956)
   Building prompt with 5 articles...
   Generating answer...
   ✅ Done in 27795ms

💬 TRẢ LỜI (27795ms):
──────────────────────────────────────────────────────────────────────
🟢 **Độ tin cậy: HIGH** (score: 0.956)

Theo Điều 655 và Điều 213 BLDS 2015, khi bạn đang xin ly hôn nhưng chưa được hoặc đã được Tòa án cho ly hôn bằng bản án hoặc quyết định chưa có hiệu lực pháp luật, tài sản chung của vợ chồng sẽ được chia như sau:

- Nếu di chúc của vợ chồng không phân chia cụ thể tài sản chung, di sản sẽ được chia đều cho cả hai người.
- Nếu di chúc phân chia cụ thể tài sản chung theo tỷ lệ, tỷ lệ này được tính trên giá trị khối di sản đang còn vào thời điểm phân chia di sản.

Nhưng请注意，根据上述条款，当您正在申请离婚但尚未获得或

In [12]:
# ============================================================
# CELL 8 V2: RAG PIPELINE - FIX HALLUCINATION + TIẾNG TRUNG + CONFIDENCE
# ============================================================
from typing import List, Dict, Tuple
import time
import re

# === SYSTEM PROMPT MỚI: STRICT hơn, force tiếng Việt ===
SYSTEM_PROMPT = """Bạn là trợ lý pháp luật chuyên về Bộ luật Dân sự Việt Nam 2015.

YÊU CẦU TUYỆT ĐỐI:
1. CHỈ trả lời bằng TIẾNG VIỆT. KHÔNG dùng tiếng Trung, tiếng Anh, hay bất kỳ ngôn ngữ nào khác.
2. CHỈ sử dụng thông tin từ các Điều luật trong phần CONTEXT bên dưới.
3. ĐỌC KỸ TIÊU ĐỀ của từng Điều luật để chọn Điều phù hợp NHẤT với câu hỏi. Không phải Điều nào có từ khóa cũng dùng được.
4. Trả lời NGẮN GỌN trong 3-6 câu, đi thẳng vào câu hỏi.
5. LUÔN trích dẫn số Điều cụ thể: "Theo Điều X BLDS 2015..."
6. Nếu các Điều trong CONTEXT KHÔNG trả lời được câu hỏi → nói rõ "Các Điều luật được cung cấp không đề cập trực tiếp đến vấn đề này" và DỪNG, không bịa.
7. KHÔNG tự mâu thuẫn trong cùng câu trả lời."""


def build_rag_prompt(query: str, retrieved_articles: List[Dict]) -> str:
    """Build prompt với context có đánh số rõ ràng để LLM dễ pick"""

    context_parts = []
    for i, art in enumerate(retrieved_articles, 1):
        context_parts.append(
            f"[{i}] {art['dieu']}: {art['title']}\n"
            f"Thuộc: {art['chuong_title']}\n"
            f"Nội dung: {art['content'].strip()}"
        )
    context = "\n\n".join(context_parts)

    user_message = f"""CONTEXT - Các Điều luật được tìm thấy:

{context}

═══════════════════════════════════════════════════════
CÂU HỎI: {query}
═══════════════════════════════════════════════════════

Phân tích từng Điều trên xem Điều nào trả lời TRỰC TIẾP câu hỏi.
- Nếu có Điều phù hợp: trả lời ngắn gọn dựa vào Điều đó.
- Nếu không có Điều nào trả lời trực tiếp: nói rõ "Các Điều luật trên không trả lời trực tiếp câu hỏi này".

CHÚ Ý: Chỉ trả lời bằng TIẾNG VIỆT."""

    return user_message


def evaluate_confidence(retrieved_articles: List[Dict]) -> Tuple[str, float, str]:
    if not retrieved_articles:
        return "low", 0.0, "🔴"
    top_score = retrieved_articles[0]["rerank_score"]
    if top_score >= config.confidence_high:
        return "high", top_score, "🟢"
    elif top_score >= config.confidence_medium:
        return "medium", top_score, "🟡"
    else:
        return "low", top_score, "🔴"


def detect_cross_domain(query: str) -> List[str]:
    query_lower = query.lower()
    return [law for law, kws in CROSS_DOMAIN_KEYWORDS.items()
            if any(kw in query_lower for kw in kws)]


def has_non_vietnamese(text: str) -> bool:
    """Detect tiếng Trung / Nhật / Hàn trong response"""
    # CJK Unified Ideographs ranges
    for char in text:
        code = ord(char)
        if (0x4E00 <= code <= 0x9FFF or   # Chinese
            0x3040 <= code <= 0x309F or   # Hiragana
            0x30A0 <= code <= 0x30FF or   # Katakana
            0xAC00 <= code <= 0xD7AF):    # Korean
            return True
    return False


def generate_answer(
    query: str,
    max_new_tokens: int = 400,
    temperature: float = 0.1,  # Giảm xuống để giảm hallucination + tránh ngôn ngữ lạ
    verbose: bool = True
) -> Dict:
    t0 = time.time()

    # ===== 1. RETRIEVE =====
    if verbose:
        print(f"🔍 Retrieving...")
    retrieved = retrieve(query, verbose=False)

    # ===== 2. EVALUATE CONFIDENCE =====
    conf_level, conf_score, conf_emoji = evaluate_confidence(retrieved)
    if verbose:
        print(f"   Confidence: {conf_emoji} {conf_level.upper()} (score: {conf_score:.3f})")
        print(f"   Top retrieved:")
        for i, art in enumerate(retrieved[:3], 1):
            print(f"      {i}. [{art['dieu']}] {art['title'][:60]} (rerank={art['rerank_score']:.2f})")

    # ===== 3. CHECK LOW CONFIDENCE =====
    if conf_level == "low":
        return {
            "answer": (
                f"🔴 Tôi không tìm thấy thông tin đủ tin cậy trong BLDS 2015 cho câu hỏi này "
                f"(score: {conf_score:.3f}).\n\n"
                f"Gợi ý: diễn đạt cụ thể hơn, hoặc câu hỏi có thể thuộc luật khác.\n\n"
                f"{LEGAL_DISCLAIMER}"
            ),
            "confidence": conf_level,
            "confidence_score": conf_score,
            "retrieved_articles": retrieved,
            "related_laws": [],
            "took_ms": int((time.time() - t0) * 1000),
        }

    # ===== 4. BUILD PROMPT =====
    if verbose:
        print(f"   Building prompt with {len(retrieved)} articles...")
    user_message = build_rag_prompt(query, retrieved)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # ===== 5. GENERATE =====
    if verbose:
        print(f"   Generating...")

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to("cuda")

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # GREEDY để deterministic + nhanh hơn
            repetition_penalty=1.15,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    answer_text = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    # ===== 6. POST-PROCESS: CẮT TIẾNG TRUNG =====
    if has_non_vietnamese(answer_text):
        # Tìm vị trí đầu tiên xuất hiện ký tự CJK
        for i, char in enumerate(answer_text):
            code = ord(char)
            if (0x4E00 <= code <= 0x9FFF or 0x3040 <= code <= 0x309F or
                0x30A0 <= code <= 0x30FF or 0xAC00 <= code <= 0xD7AF):
                answer_text = answer_text[:i].strip()
                break
        if verbose:
            print(f"   ⚠️  Đã cắt phần ngôn ngữ không phải tiếng Việt")

    # ===== 7. FORMAT FINAL =====
    related_laws = detect_cross_domain(query)

    final_answer = f"{conf_emoji} **Độ tin cậy: {conf_level.upper()}** (score: {conf_score:.3f})\n\n"
    final_answer += answer_text + "\n\n"
    final_answer += "📚 **Tham khảo các Điều liên quan:**\n"
    for art in retrieved[:3]:
        final_answer += f"   • {art['dieu']}: {art['title']}\n"

    if related_laws:
        final_answer += f"\n⚠️ **Có thể liên quan luật khác:**\n"
        for law in related_laws:
            final_answer += f"   • {law}\n"

    final_answer += f"\n{LEGAL_DISCLAIMER}"

    took_ms = int((time.time() - t0) * 1000)
    if verbose:
        print(f"   ✅ Done in {took_ms}ms\n")

    return {
        "answer": final_answer,
        "confidence": conf_level,
        "confidence_score": conf_score,
        "retrieved_articles": retrieved,
        "related_laws": related_laws,
        "took_ms": took_ms,
    }


# ===== TEST V2 =====
print("=" * 70)
print("🧪 TEST RAG PIPELINE V2 (fix hallucination + tiếng Trung)")
print("=" * 70)

test_questions = [
    "Tài sản chung của vợ chồng được quy định ra sao trong BLDS?",
    "Người chưa thành niên có thể tự mình ký hợp đồng không?",
    "Hợp đồng vô hiệu là gì?",
    "Thừa kế theo pháp luật là gì?",
]

for q in test_questions:
    print(f"\n{'═' * 70}")
    print(f"❓ {q}")
    print(f"{'═' * 70}")
    result = generate_answer(q, verbose=True)
    print(f"💬 TRẢ LỜI ({result['took_ms']}ms):")
    print(f"{'─' * 70}")
    print(result["answer"])

🧪 TEST RAG PIPELINE V2 (fix hallucination + tiếng Trung)

══════════════════════════════════════════════════════════════════════
❓ Tài sản chung của vợ chồng được quy định ra sao trong BLDS?
══════════════════════════════════════════════════════════════════════
🔍 Retrieving...


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


   Confidence: 🟢 HIGH (score: 0.767)
   Top retrieved:
      1. [Điều 213] Sở hữu chung của vợ chồng (rerank=0.77)
      2. [Điều 655] Việc thừa kế trong trường hợp vợ, chồng đã chia tài sản chun (rerank=0.05)
      3. [Điều 212] Sở hữu chung của các thành viên gia đình (rerank=0.04)
   Building prompt with 5 articles...
   Generating...
   ✅ Done in 4071ms

💬 TRẢ LỜI (4071ms):
──────────────────────────────────────────────────────────────────────
🟢 **Độ tin cậy: HIGH** (score: 0.767)

Theo Điều 213 BLDS 2015...

📚 **Tham khảo các Điều liên quan:**
   • Điều 213: Sở hữu chung của vợ chồng
   • Điều 655: Việc thừa kế trong trường hợp vợ, chồng đã chia tài sản chung; vợ, chồng đang xin ly hôn hoặc đã kết hôn với người khác
   • Điều 212: Sở hữu chung của các thành viên gia đình

⚠️ **Có thể liên quan luật khác:**
   • Luật Hôn nhân và Gia đình 2014

⚠️ KHUYẾN CÁO: Thông tin chỉ mang tính THAM KHẢO dựa trên BLDS 2015. Không thay thế tư vấn pháp lý chuyên nghiệp. Hotline trợ giúp pháp lý m

In [13]:
# ============================================================
# CELL 8 V3: RAG PIPELINE - FIX CỤT + FEW-SHOT + BALANCED STRICTNESS
# ============================================================
from typing import List, Dict, Tuple
import time

# === SYSTEM PROMPT V3: Cân bằng giữa strict và helpful ===
SYSTEM_PROMPT = """Bạn là trợ lý pháp luật chuyên về Bộ luật Dân sự Việt Nam 2015.

NGUYÊN TẮC:
1. Trả lời bằng TIẾNG VIỆT, KHÔNG dùng tiếng nước ngoài.
2. Đọc kỹ TIÊU ĐỀ và NỘI DUNG các Điều luật trong CONTEXT để chọn Điều phù hợp nhất.
3. Trả lời ĐẦY ĐỦ trong 4-8 câu, giải thích rõ ràng nội dung Điều luật.
4. LUÔN trích dẫn số Điều cụ thể VÀ TRÍCH NỘI DUNG cụ thể của Điều đó (không chỉ nói "Theo Điều X...").
5. Nếu Điều luật có nhiều khoản, liệt kê các khoản chính.
6. Tuyệt đối KHÔNG bịa thông tin ngoài CONTEXT."""


# === FEW-SHOT EXAMPLES (DEMONSTRATE cách trả lời tốt) ===
FEW_SHOT_EXAMPLE = """VÍ DỤ về cách trả lời TỐT:

Câu hỏi mẫu: "Cá nhân có quyền dân sự gì?"
Context mẫu có Điều 16: "Cá nhân có năng lực pháp luật dân sự đầy đủ, trừ trường hợp..."

Trả lời mẫu (TỐT):
Theo Điều 16 BLDS 2015, cá nhân có năng lực pháp luật dân sự, là khả năng có quyền và nghĩa vụ dân sự. Năng lực này có từ khi sinh ra và chấm dứt khi chết. Mọi cá nhân đều bình đẳng về năng lực pháp luật dân sự, không bị phân biệt bởi dân tộc, giới tính, tôn giáo, hoàn cảnh kinh tế.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
GIỜ ĐẾN CÂU HỎI CỦA BẠN:
"""


def build_rag_prompt(query: str, retrieved_articles: List[Dict]) -> str:
    """Build prompt với few-shot example + context"""

    context_parts = []
    for i, art in enumerate(retrieved_articles, 1):
        # Cắt content quá dài để tránh prompt quá lớn
        content = art['content'].strip()
        if len(content) > 800:
            content = content[:800] + "..."

        context_parts.append(
            f"[Điều luật {i}]\n"
            f"{art['dieu']}: {art['title']}\n"
            f"(Thuộc: {art['chuong_title']})\n"
            f"Nội dung: {content}"
        )
    context = "\n\n".join(context_parts)

    user_message = f"""{FEW_SHOT_EXAMPLE}

CONTEXT - Các Điều luật được tìm thấy:

{context}

═══════════════════════════════════════════════════════
CÂU HỎI THỰC TẾ: {query}
═══════════════════════════════════════════════════════

Hãy trả lời câu hỏi trên dựa vào các Điều luật trong CONTEXT.
- Chọn 1-2 Điều phù hợp NHẤT để trả lời.
- Trích dẫn cụ thể số Điều VÀ nội dung của Điều đó.
- Trả lời đầy đủ trong 4-8 câu bằng tiếng Việt."""

    return user_message


def evaluate_confidence(retrieved_articles: List[Dict]) -> Tuple[str, float, str]:
    if not retrieved_articles:
        return "low", 0.0, "🔴"
    top_score = retrieved_articles[0]["rerank_score"]
    if top_score >= config.confidence_high:
        return "high", top_score, "🟢"
    elif top_score >= config.confidence_medium:
        return "medium", top_score, "🟡"
    else:
        return "low", top_score, "🔴"


def detect_cross_domain(query: str) -> List[str]:
    query_lower = query.lower()
    return [law for law, kws in CROSS_DOMAIN_KEYWORDS.items()
            if any(kw in query_lower for kw in kws)]


def has_non_vietnamese(text: str) -> bool:
    for char in text:
        code = ord(char)
        if (0x4E00 <= code <= 0x9FFF or 0x3040 <= code <= 0x309F or
            0x30A0 <= code <= 0x30FF or 0xAC00 <= code <= 0xD7AF):
            return True
    return False


def generate_answer(
    query: str,
    max_new_tokens: int = 500,
    verbose: bool = True
) -> Dict:
    t0 = time.time()

    # ===== 1. RETRIEVE =====
    if verbose:
        print(f"🔍 Retrieving...")
    retrieved = retrieve(query, verbose=False)

    conf_level, conf_score, conf_emoji = evaluate_confidence(retrieved)
    if verbose:
        print(f"   Confidence: {conf_emoji} {conf_level.upper()} (score: {conf_score:.3f})")
        print(f"   Top retrieved:")
        for i, art in enumerate(retrieved[:3], 1):
            print(f"      {i}. [{art['dieu']}] {art['title'][:60]} (rerank={art['rerank_score']:.2f})")

    if conf_level == "low":
        return {
            "answer": (
                f"🔴 Không tìm thấy thông tin đủ tin cậy trong BLDS 2015 (score: {conf_score:.3f}).\n\n"
                f"{LEGAL_DISCLAIMER}"
            ),
            "confidence": conf_level,
            "confidence_score": conf_score,
            "retrieved_articles": retrieved,
            "related_laws": [],
            "took_ms": int((time.time() - t0) * 1000),
        }

    # ===== 4. BUILD PROMPT =====
    user_message = build_rag_prompt(query, retrieved)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # ===== 5. GENERATE với sampling cân bằng =====
    if verbose:
        print(f"   Generating...")

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=6144).to("cuda")

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            min_new_tokens=80,           # ✅ FIX CỤT: bắt buộc ít nhất 80 tokens
            do_sample=True,              # ✅ Sampling lại để LLM nói nhiều hơn
            temperature=0.3,             # Thấp để giảm hallucination
            top_p=0.85,
            repetition_penalty=1.15,
            no_repeat_ngram_size=4,      # ✅ Tránh lặp 4-gram
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    answer_text = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    # Cắt tiếng Trung nếu có
    if has_non_vietnamese(answer_text):
        for i, char in enumerate(answer_text):
            code = ord(char)
            if (0x4E00 <= code <= 0x9FFF or 0x3040 <= code <= 0x309F or
                0x30A0 <= code <= 0x30FF or 0xAC00 <= code <= 0xD7AF):
                answer_text = answer_text[:i].strip()
                break

    # ===== 7. FORMAT =====
    related_laws = detect_cross_domain(query)

    final_answer = f"{conf_emoji} **Độ tin cậy: {conf_level.upper()}** (score: {conf_score:.3f})\n\n"
    final_answer += answer_text + "\n\n"
    final_answer += "📚 **Tham khảo các Điều liên quan:**\n"
    for art in retrieved[:3]:
        final_answer += f"   • {art['dieu']}: {art['title']}\n"

    if related_laws:
        final_answer += f"\n⚠️ **Có thể liên quan luật khác:**\n"
        for law in related_laws:
            final_answer += f"   • {law}\n"

    final_answer += f"\n{LEGAL_DISCLAIMER}"

    took_ms = int((time.time() - t0) * 1000)
    if verbose:
        print(f"   ✅ Done in {took_ms}ms\n")

    return {
        "answer": final_answer,
        "confidence": conf_level,
        "confidence_score": conf_score,
        "retrieved_articles": retrieved,
        "related_laws": related_laws,
        "took_ms": took_ms,
    }


# ===== TEST V3 =====
print("=" * 70)
print("🧪 TEST V3 - FIX CỤT + FEW-SHOT")
print("=" * 70)

test_questions = [
    "Thừa kế theo pháp luật là gì?",
    "Hợp đồng vô hiệu là gì?",
    "Người chưa thành niên có thể tự mình ký hợp đồng không?",
    "Tài sản chung của vợ chồng được quy định ra sao?",
]

for q in test_questions:
    print(f"\n{'═' * 70}")
    print(f"❓ {q}")
    print(f"{'═' * 70}")
    result = generate_answer(q, verbose=True)
    print(f"💬 TRẢ LỜI ({result['took_ms']}ms):")
    print(f"{'─' * 70}")
    print(result["answer"])

🧪 TEST V3 - FIX CỤT + FEW-SHOT

══════════════════════════════════════════════════════════════════════
❓ Thừa kế theo pháp luật là gì?
══════════════════════════════════════════════════════════════════════
🔍 Retrieving...
   Confidence: 🟢 HIGH (score: 0.999)
   Top retrieved:
      1. [Điều 649] Thừa kế theo pháp luật (rerank=1.00)
      2. [Điều 651] Người thừa kế theo pháp luật (rerank=0.99)
      3. [Điều 650] Những trường hợp thừa kế theo pháp luật (rerank=0.98)
   Generating...
   ✅ Done in 44713ms

💬 TRẢ LỜI (44713ms):
──────────────────────────────────────────────────────────────────────
🟢 **Độ tin cậy: HIGH** (score: 0.999)

Theo Điều luật 647 và 648 BLDS 1995, thay thế cho Điều 645 và 652 BLDS 06/2010 như vậy, thừa kế thep pháp luât là một hình thức thừa kế mà pháp luật quy Định về hàng thừa kế và trình tự, điều kiện thừa kế. 

Thứ nhất, theo Điều 653 BLDS 6/2100, hàng thừa kế thephapluat gồm những người thuộc hàng thừa kế thứ Nhất, thứ Hai và thứ Ba. Hàng thừa kế Thứ nhất bao

# **tính trên bước này**

 ============================================================
# CELL 5: CHUNKING + EMBEDDING + VECTOR STORE
# ============================================================
# Mục đích: Chia nhỏ văn bản → vector hóa → lưu vào database
#
# Quy trình:
#   Articles (689 Điều) → Chunks (mỗi Điều = 1 chunk, Điều dài chia theo Khoản)
#   → Embedding (multilingual-e5-large, 1024 chiều)
#   → ChromaDB (vector database, cosine similarity)
#
# Tại sao mỗi Điều = 1 chunk:
#   Trong pháp luật, mỗi Điều là đơn vị ngữ nghĩa hoàn chỉnh.
#   Chia nhỏ hơn (theo câu) sẽ mất ngữ cảnh.

In [ ]:

# ============================================================
# CELL 5: CHUNKING + EMBEDDING + VECTOR STORE
# ============================================================
# Mục đích: Chia nhỏ văn bản → vector hóa → lưu vào database
#
# Quy trình:
#   Articles (689 Điều) → Chunks (mỗi Điều = 1 chunk, Điều dài chia theo Khoản)
#   → Embedding (multilingual-e5-large, 1024 chiều)
#   → ChromaDB (vector database, cosine similarity)
#
# Tại sao mỗi Điều = 1 chunk:
#   Trong pháp luật, mỗi Điều là đơn vị ngữ nghĩa hoàn chỉnh.
#   Chia nhỏ hơn (theo câu) sẽ mất ngữ cảnh.

from sentence_transformers import SentenceTransformer, CrossEncoder
import chromadb

@dataclass
class LegalChunk:
    """Đại diện cho 1 đoạn văn bản pháp luật đã được chia nhỏ"""
    text: str           # Nội dung text
    dieu: str           # "Điều 35"
    dieu_number: int    # 35
    title: str          # Tên Điều
    chuong: str         # Chương chứa Điều này
    phan: str           # Phần chứa Điều này
    chunk_id: str       # ID duy nhất (VD: "d035" hoặc "d035_k1")
    metadata: Dict = field(default_factory=dict)  # Metadata bổ sung


def build_chunks(articles: List[Dict]) -> List[LegalChunk]:
    """
    Chia articles thành chunks.
    Quy tắc:
    - Điều ngắn (< chunk_size) → giữ nguyên = 1 chunk
    - Điều dài (> chunk_size) → chia theo Khoản (1., 2., 3.,...)
    """
    chunks = []
    for art in articles:
        content = art.get("content", "")
        if not content or len(content.strip()) < 20:
            continue

        # Metadata cho citation (trích dẫn nguồn)
        meta = {
            "dieu": art["dieu"], "dieu_number": art.get("dieu_number", 0),
            "title": art.get("title", ""), "chuong": art.get("chuong", ""),
            "phan": art.get("phan", ""), "source": SCOPE,
            "effective_date": "01/01/2017",
            "amended": art.get("amended", False),
            "amended_info": json.dumps(art.get("amended_info", {}), ensure_ascii=False) if art.get("amended_info") else "",
            "citation": f"{art['dieu']}. {art.get('title','')} - {SCOPE}",
        }

        if len(content) <= config.chunk_size:
            # Điều ngắn → 1 chunk
            chunks.append(LegalChunk(
                text=content.strip(), dieu=art["dieu"],
                dieu_number=art.get("dieu_number", 0),
                title=art.get("title", ""), chuong=art.get("chuong", ""),
                phan=art.get("phan", ""),
                chunk_id=f"d{art.get('dieu_number',0):03d}", metadata=meta
            ))
        else:
            # Điều dài → chia theo Khoản (pattern: "\n1. ", "\n2. ",...)
            parts = re.split(r'(?=\n\d+\.\s)', content)
            for i, part in enumerate(parts):
                if part.strip() and len(part.strip()) > 20:
                    chunks.append(LegalChunk(
                        text=part.strip(), dieu=art["dieu"],
                        dieu_number=art.get("dieu_number", 0),
                        title=art.get("title", ""), chuong=art.get("chuong", ""),
                        phan=art.get("phan", ""),
                        chunk_id=f"d{art.get('dieu_number',0):03d}_k{i}",
                        metadata={**meta, "sub_part": i}
                    ))
    return chunks


# === Tạo chunks ===
legal_chunks = build_chunks(civil_code_articles)
print(f"✂️ {len(civil_code_articles)} điều → {len(legal_chunks)} chunks")

# === Load models ===
print("📥 Loading embedding model (multilingual-e5-large)...")
# Model này chuyển text → vector 1024 chiều, hỗ trợ tiếng Việt tốt
embedding_model = SentenceTransformer(config.embedding_model_name)

print("📥 Loading reranker (bge-reranker-base)...")
# Cross-encoder: so sánh trực tiếp (query, document) → score chính xác hơn
reranker = CrossEncoder(config.reranker_model_name, max_length=512)
print(f"   ✅ Embedding dim={embedding_model.get_sentence_embedding_dimension()}")

# === Tạo ChromaDB (vector database) ===
# ChromaDB lưu vectors + metadata, hỗ trợ similarity search
chroma_client = chromadb.PersistentClient(path=config.chroma_path)  # Lưu trên Drive
try: chroma_client.delete_collection("blds2015")  # Xóa cũ nếu có
except: pass
collection = chroma_client.create_collection("blds2015", metadata={"hnsw:space": "cosine"})

# === Embed tất cả chunks và lưu vào ChromaDB ===
# Format "passage: {text}" là yêu cầu của model e5 (phân biệt query vs passage)
BATCH = 64  # Embed 64 chunks cùng lúc (nhanh hơn từng cái)
for i in tqdm(range(0, len(legal_chunks), BATCH), desc="Embedding & Indexing"):
    batch = legal_chunks[i:i+BATCH]
    texts = [f"passage: {c.text}" for c in batch]  # Format e5
    embs = embedding_model.encode(texts, normalize_embeddings=True, show_progress_bar=False)
    collection.add(
        ids=[c.chunk_id for c in batch],
        embeddings=embs.tolist(),
        documents=[c.text for c in batch],
        metadatas=[c.metadata for c in batch]
    )
print(f"✅ Vector store: {collection.count()} chunks indexed")



# ============================================================
# CELL 6: [FEATURE 2] QUERY REFORMULATION (HyDE)
# ============================================================
# Mục đích: Viết lại câu hỏi công dân → ngôn ngữ pháp lý
#
# Vấn đề: Công dân hỏi "nhà tôi bị chiếm" nhưng trong luật ghi
#          "chiếm hữu không có căn cứ pháp luật" (Điều 165)
#          → Nếu search trực tiếp sẽ không tìm được!
#
# Giải pháp: 2 bước
#   Bước 1 (Rule-based): Mapping từ đời thường → thuật ngữ pháp lý
#   Bước 2 (LLM-based): Dùng model viết lại câu hỏi (nếu có model)
#
# Tham khảo: Gao et al., 2022 "Precise Zero-Shot Dense Retrieval without
#            Relevance Labels" (HyDE - Hypothetical Document Embedding)

In [ ]:
# ============================================================
# CELL 6: [FEATURE 2] QUERY REFORMULATION (HyDE)
# ============================================================
# Mục đích: Viết lại câu hỏi công dân → ngôn ngữ pháp lý
#
# Vấn đề: Công dân hỏi "nhà tôi bị chiếm" nhưng trong luật ghi
#          "chiếm hữu không có căn cứ pháp luật" (Điều 165)
#          → Nếu search trực tiếp sẽ không tìm được!
#
# Giải pháp: 2 bước
#   Bước 1 (Rule-based): Mapping từ đời thường → thuật ngữ pháp lý
#   Bước 2 (LLM-based): Dùng model viết lại câu hỏi (nếu có model)
#
# Tham khảo: Gao et al., 2022 "Precise Zero-Shot Dense Retrieval without
#            Relevance Labels" (HyDE - Hypothetical Document Embedding)

# Bảng mapping: ngôn ngữ đời thường → thuật ngữ pháp lý BLDS 2015
LEGAL_TERM_MAPPING = {
    # --- Tài sản ---
    "nhà": "bất động sản, nhà ở, quyền sở hữu nhà",
    "đất": "quyền sử dụng đất, bất động sản",
    "xe": "động sản, tài sản",
    "tiền": "tài sản, nghĩa vụ tài chính, khoản nợ",
    "nợ": "nghĩa vụ dân sự, nghĩa vụ trả nợ",
    "vay": "hợp đồng vay tài sản, nghĩa vụ trả nợ",
    # --- Hợp đồng ---
    "mua bán": "hợp đồng mua bán tài sản",
    "thuê": "hợp đồng thuê tài sản",
    "cho mượn": "hợp đồng mượn tài sản",
    "đặt cọc": "đặt cọc, biện pháp bảo đảm",
    "hứa": "cam kết, giao dịch dân sự",
    # --- Thừa kế ---
    "di chúc": "di chúc, thừa kế theo di chúc",
    "chia tài sản": "phân chia di sản, thừa kế",
    "người chết": "người để lại di sản, mở thừa kế",
    "con cái": "người thừa kế theo pháp luật, hàng thừa kế",
    # --- Bồi thường ---
    "đền bù": "bồi thường thiệt hại",
    "tai nạn": "bồi thường thiệt hại ngoài hợp đồng",
    "hư hỏng": "thiệt hại về tài sản",
    # --- Quan hệ ---
    "hàng xóm": "quyền đối với bất động sản liền kề",
    "lấn chiếm": "chiếm hữu không có căn cứ pháp luật",
    "tranh chấp": "tranh chấp dân sự, khởi kiện",
}


def reformulate_query(query: str, model=None, tokenizer=None) -> str:
    """
    Viết lại câu hỏi để retrieval chính xác hơn.

    Ví dụ:
        Input:  "nhà tôi bị hàng xóm lấn chiếm"
        Output: "nhà tôi bị hàng xóm lấn chiếm (bất động sản, quyền đối với
                 bất động sản liền kề | chiếm hữu không có căn cứ pháp luật)"
    """
    # === Bước 1: Rule-based expansion (nhanh, không cần GPU) ===
    expanded_terms = []
    q_lower = query.lower()
    for term, legal_term in LEGAL_TERM_MAPPING.items():
        if term in q_lower:
            expanded_terms.append(legal_term)

    if expanded_terms:
        expansion = " | ".join(expanded_terms[:3])  # Tối đa 3 terms
        expanded_query = f"{query} ({expansion})"
    else:
        expanded_query = query

    # === Bước 2: LLM reformulation (chính xác hơn, cần GPU) ===
    if model is not None and tokenizer is not None:
        try:
            prompt = f"""<|im_start|>system
Viết lại câu hỏi sau thành ngôn ngữ pháp lý theo BLDS 2015. Chỉ trả về câu hỏi mới.
<|im_end|>
<|im_start|>user
{query}
<|im_end|>
<|im_start|>assistant
"""
            inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=100, temperature=0.3,
                                     do_sample=False, pad_token_id=tokenizer.pad_token_id)
            reformulated = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            reformulated = reformulated.split("<|im_end|>")[0].strip()
            if reformulated and len(reformulated) > 10:
                return f"{query} | {reformulated}"  # Kết hợp cả 2 để tăng recall
        except:
            pass

    return expanded_query



# ============================================================
# CELL 7: RETRIEVAL + RERANKING + CONFIDENCE + AMBIGUITY
# ============================================================
# Mục đích: Tìm kiếm documents liên quan + đánh giá độ tin cậy
#
# Pipeline:
#   Query → Reformulate → Embed → Vector Search (top-15)
#                                       ↓
#                              Cross-encoder Rerank (top-5)
#                                       ↓
#                              Confidence Scoring (high/medium/low)
#                                       ↓
#                              Ambiguity Check (nhiều Chương → hỏi lại)

In [ ]:

# ============================================================
# CELL 7: RETRIEVAL + RERANKING + CONFIDENCE + AMBIGUITY
# ============================================================
# Mục đích: Tìm kiếm documents liên quan + đánh giá độ tin cậy
#
# Pipeline:
#   Query → Reformulate → Embed → Vector Search (top-15)
#                                       ↓
#                              Cross-encoder Rerank (top-5)
#                                       ↓
#                              Confidence Scoring (high/medium/low)
#                                       ↓
#                              Ambiguity Check (nhiều Chương → hỏi lại)

def retrieve_raw(query: str, top_k: int = None) -> List[Dict]:
    """
    Bước 1: Vector similarity search trong ChromaDB.
    Dùng cosine similarity giữa query embedding và document embeddings.
    Trả về top-K documents gần nhất.
    """
    if top_k is None:
        top_k = config.top_k_retrieval

    # Embed query (format "query: {text}" theo yêu cầu của e5)
    q_emb = embedding_model.encode([f"query: {query}"], normalize_embeddings=True).tolist()

    # Search trong ChromaDB
    n = min(top_k, collection.count())
    if n == 0:
        return []
    results = collection.query(query_embeddings=q_emb, n_results=n,
                               include=["documents", "metadatas", "distances"])

    # Format kết quả
    docs = []
    for i in range(len(results["ids"][0])):
        docs.append({
            "id": results["ids"][0][i],
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "similarity": 1 - results["distances"][0][i]  # distance → similarity
        })
    return docs


def rerank_docs(query: str, docs: List[Dict], top_k: int = None) -> List[Dict]:
    """
    Bước 2: Cross-encoder reranking.
    Tại sao cần: Vector search nhanh nhưng không chính xác bằng cross-encoder.
    Cross-encoder so sánh TRỰC TIẾP (query, document) → score chính xác hơn.
    Trade-off: Chậm hơn (nên chỉ rerank top-15, không phải toàn bộ corpus).
    """
    if top_k is None:
        top_k = config.top_k_rerank
    if not docs:
        return []

    # Tạo pairs [query, document] cho cross-encoder
    pairs = [[query, d["text"]] for d in docs]
    scores = reranker.predict(pairs)  # Score cho mỗi pair

    # Gán score và sort giảm dần
    for i, d in enumerate(docs):
        d["rerank_score"] = float(scores[i])
    return sorted(docs, key=lambda x: x["rerank_score"], reverse=True)[:top_k]


def full_retrieve(query: str, model=None, tokenizer=None) -> Tuple[List[Dict], float, str, str, bool]:
    """
    Full retrieval pipeline với tất cả features.

    Returns:
        - docs: List documents đã rerank
        - confidence: Float 0-1 (độ tin cậy)
        - level: "high" / "medium" / "low"
        - reformulated: Câu hỏi đã viết lại
        - is_ambiguous: True nếu kết quả đến từ nhiều Chương (đa nghĩa)
    """
    # [FEATURE 2] Viết lại câu hỏi
    reformulated = reformulate_query(query, model, tokenizer)

    # Retrieve với CẢ query gốc VÀ query đã reformulate
    docs_original = retrieve_raw(query)
    docs_reformulated = retrieve_raw(reformulated) if reformulated != query else []

    # Merge & loại bỏ trùng lặp (giữ unique documents)
    seen_ids = set()
    merged = []
    for d in docs_original + docs_reformulated:
        if d["id"] not in seen_ids:
            seen_ids.add(d["id"])
            merged.append(d)

    # Rerank merged results
    reranked = rerank_docs(query, merged)

    # === CONFIDENCE SCORING ===
    # Ý tưởng: Nếu top documents có score cao → chatbot tự tin trả lời
    #           Nếu score thấp → từ chối (an toàn cho công dân)
    if not reranked:
        return [], 0.0, "low", reformulated, False

    # Weighted average của top-3 scores (top-1 quan trọng nhất)
    top_scores = [d["rerank_score"] for d in reranked[:3]]
    weights = [0.5, 0.3, 0.2][:len(top_scores)]
    raw_conf = sum(s * w for s, w in zip(top_scores, weights)) / sum(weights[:len(top_scores)])
    # Normalize từ [-1,1] → [0,1] (rerank scores có thể âm)
    confidence = max(0, min(1, (raw_conf + 1) / 2))

    # Phân loại confidence level
    if confidence >= config.confidence_high:
        level = "high"      # 🟢 Trả lời đầy đủ
    elif confidence >= config.confidence_medium:
        level = "medium"    # 🟡 Trả lời + cảnh báo
    else:
        level = "low"       # 🔴 TỪ CHỐI trả lời

    # === [FEATURE 3] AMBIGUITY DETECTION ===
    # Nếu top-5 results đến từ >= 3 Chương khác nhau → câu hỏi đa nghĩa
    chapters = set()
    for d in reranked[:5]:
        ch = d.get("metadata", {}).get("chuong", "")
        if ch:
            chapters.add(ch)
    is_ambiguous = len(chapters) >= config.ambiguity_chapter_threshold

    return reranked, confidence, level, reformulated, is_ambiguous


# === Test nhanh ===
test_docs, test_conf, test_lvl, test_ref, test_amb = full_retrieve("nhà tôi bị hàng xóm lấn chiếm")
print(f"🧪 Test: conf={test_conf:.3f} ({test_lvl}), reformulated='{test_ref[:60]}...', ambiguous={test_amb}")

# ============================================================
# CELL 8: SINH Q&A DATASET + FINE-TUNING QLoRA
# ============================================================
# Mục đích:
#   1. Dùng LLM sinh câu hỏi-trả lời từ corpus (Self-Instruct method)
#   2. Fine-tune model với QLoRA để trả lời tốt hơn cho domain pháp luật
#
# Tại sao cần fine-tuning:
#   - Model gốc (Qwen) biết nhiều thứ nhưng không chuyên về luật VN
#   - Sau fine-tuning: trả lời chính xác hơn, trích dẫn Điều/Khoản tốt hơn
#
# QLoRA (Dettmers et al., 2023):
#   - Quantize model xuống 4-bit (tiết kiệm VRAM: 28GB → 5GB)
#   - Chỉ train thêm LoRA adapters (~0.5% tham số)
#   - Kết quả gần bằng full fine-tuning nhưng chạy được trên T4 16GB

In [ ]:

# ============================================================
# CELL 8: SINH Q&A DATASET + FINE-TUNING QLoRA
# ============================================================
# Mục đích:
#   1. Dùng LLM sinh câu hỏi-trả lời từ corpus (Self-Instruct method)
#   2. Fine-tune model với QLoRA để trả lời tốt hơn cho domain pháp luật
#
# Tại sao cần fine-tuning:
#   - Model gốc (Qwen) biết nhiều thứ nhưng không chuyên về luật VN
#   - Sau fine-tuning: trả lời chính xác hơn, trích dẫn Điều/Khoản tốt hơn
#
# QLoRA (Dettmers et al., 2023):
#   - Quantize model xuống 4-bit (tiết kiệm VRAM: 28GB → 5GB)
#   - Chỉ train thêm LoRA adapters (~0.5% tham số)
#   - Kết quả gần bằng full fine-tuning nhưng chạy được trên T4 16GB

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import Dataset

# --- Load model 4-bit ---
print("📥 Loading Qwen2.5-7B (4-bit quantization)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,              # Quantize xuống 4-bit (tiết kiệm 4x VRAM)
    bnb_4bit_quant_type="nf4",      # NormalFloat4 (tốt hơn FP4)
    bnb_4bit_compute_dtype=torch.float16,  # Tính toán bằng FP16
    bnb_4bit_use_double_quant=True, # Quantize cả quantization constants
)
tokenizer = AutoTokenizer.from_pretrained(config.llm_model_name, trust_remote_code=True, padding_side="right")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # Cần pad token cho training

model = AutoModelForCausalLM.from_pretrained(
    config.llm_model_name, quantization_config=bnb_config,
    device_map="auto",  # Tự động phân bổ layers lên GPU
    trust_remote_code=True, torch_dtype=torch.float16,
)

# --- System prompt (hướng dẫn model cách trả lời) ---
SYSTEM_PROMPT = f"""Bạn là trợ lý pháp luật dân sự Việt Nam. Phạm vi: {SCOPE}.
Quy tắc:
1. Trả lời DỰA TRÊN Bộ luật Dân sự 2015, trích dẫn Điều/Khoản cụ thể
2. Nếu câu hỏi NGOÀI phạm vi → từ chối, hướng dẫn đúng nguồn
3. Luôn nhắc: thông tin chỉ mang tính tham khảo
4. Dùng ngôn ngữ dễ hiểu cho công dân"""

# --- Sinh Q&A từ corpus bằng LLM ---
# Phương pháp: Cho model đọc từng Điều luật → sinh câu hỏi tự nhiên + trả lời
print("\n📝 Generating Q&A dataset from corpus...")
qa_dataset = []
real_chunks = [c for c in legal_chunks if len(c.text) > 50]

for chunk in tqdm(real_chunks[:150], desc="Generating Q&A"):
    if len(qa_dataset) >= 500:
        break
    # Prompt yêu cầu model sinh Q&A từ Điều luật
    prompt = f"""<|im_start|>system
Tạo 2 cặp câu hỏi-trả lời từ điều luật. Câu hỏi tự nhiên như công dân hỏi. Trả lời trích dẫn Điều/Khoản.
Output JSON: [{{"question":"...","answer":"..."}}]
<|im_end|>
<|im_start|>user
{chunk.metadata.get('citation','')}: {chunk.text[:1000]}
<|im_end|>
<|im_start|>assistant
"""
    try:
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1536).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=400, temperature=0.8, top_p=0.9,
                                 do_sample=True, pad_token_id=tokenizer.pad_token_id)
        resp = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        # Parse JSON từ response
        match = re.search(r'\[.*?\]', resp, re.DOTALL)
        if match:
            for item in json.loads(match.group()):
                if isinstance(item, dict) and len(item.get("question","")) > 15 and len(item.get("answer","")) > 30:
                    qa_dataset.append({**item, "citation": chunk.metadata.get("citation",""), "dieu": chunk.dieu})
    except:
        pass
    if len(qa_dataset) % 30 == 0:
        torch.cuda.empty_cache()  # Giải phóng VRAM định kỳ

print(f"✅ Generated: {len(qa_dataset)} Q&A pairs")

# --- Format training data (chat template) ---
training_data = []
for qa in qa_dataset:
    training_data.append({"messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": qa["question"]},
        {"role": "assistant", "content": f"{qa['answer']}\n\n📎 Căn cứ: {qa.get('citation','')}"}
    ]})

# --- Áp dụng LoRA adapters ---
# LoRA: Thêm ma trận nhỏ (rank=16) vào các layers attention
# Chỉ train ma trận này (~50M params) thay vì toàn bộ model (7B params)
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=config.lora_r,            # Rank = 16 (kích thước ma trận LoRA)
    lora_alpha=config.lora_alpha,  # Scaling = 32
    lora_dropout=config.lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
    # Áp dụng LoRA vào TẤT CẢ layers attention + MLP
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora_config)

# In thống kê trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"✅ LoRA: {trainable/1e6:.1f}M trainable / {total/1e9:.1f}B total ({100*trainable/total:.2f}%)")

# --- Chuẩn bị dataset ---
dataset = Dataset.from_list(training_data)
# Áp dụng chat template (format messages thành text cho training)
dataset = dataset.map(
    lambda x: {"text": tokenizer.apply_chat_template(x["messages"], tokenize=False, add_generation_prompt=False)},
    remove_columns=dataset.column_names
)
split = dataset.train_test_split(test_size=0.1, seed=42)  # 90% train, 10% eval

# --- Training ---
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=split["train"], eval_dataset=split["test"],
    max_seq_length=config.max_seq_length, dataset_text_field="text", packing=False,
    args=TrainingArguments(
        output_dir="./output",
        num_train_epochs=config.num_epochs,
        per_device_train_batch_size=config.batch_size,
        gradient_accumulation_steps=config.gradient_accumulation_steps,  # Effective batch = 16
        learning_rate=config.learning_rate,
        warmup_ratio=config.warmup_ratio,  # 10% warmup
        lr_scheduler_type="cosine",        # Cosine annealing
        logging_steps=5,                   # Log mỗi 5 steps
        eval_strategy="epoch",             # Eval cuối mỗi epoch
        save_strategy="epoch",
        fp16=True,                         # Mixed precision (nhanh hơn)
        optim="paged_adamw_8bit",          # Optimizer tiết kiệm VRAM
        max_grad_norm=0.3,                 # Gradient clipping
        gradient_checkpointing=True,       # Tiết kiệm VRAM (đổi lấy speed)
        load_best_model_at_end=True,       # Load model tốt nhất
        report_to="none",
    ),
)

print("🎯 Fine-tuning... (mất ~20-40 phút)")
result = trainer.train()

# Lưu adapter lên Drive (để lần sau không cần train lại)
model.save_pretrained(config.adapter_path)
tokenizer.save_pretrained(config.adapter_path)

# --- Vẽ biểu đồ loss ---
logs = trainer.state.log_history
tl = [(l["step"],l["loss"]) for l in logs if "loss" in l]
el = [(l["step"],l["eval_loss"]) for l in logs if "eval_loss" in l]
plt.figure(figsize=(10,4))
if tl: plt.plot(*zip(*tl),'b-',alpha=0.7,label='Train Loss')
if el: plt.plot(*zip(*el),'r-o',label='Eval Loss')
plt.xlabel('Steps'); plt.ylabel('Loss')
plt.title('QLoRA Fine-tuning Loss Curve')
plt.legend(); plt.grid(alpha=0.3)
plt.savefig(os.path.join(config.results_path,"loss.png"),dpi=150); plt.show()
print(f"✅ Training done. Final loss={result.training_loss:.4f}")
print(f"💾 Adapter saved: {config.adapter_path}")



# ============================================================
# CELL 9: CHATBOT ENGINE (TẤT CẢ FEATURES TÍCH HỢP)
# ============================================================
# Mục đích: Engine chính xử lý câu hỏi từ user
#
# Flow xử lý mỗi câu hỏi:
#   1. Kiểm tra scope (ngoài Luật Dân sự → từ chối)
#   2. Retrieve + Confidence scoring
#   3. Kiểm tra ambiguity (đa nghĩa → hỏi lại)
#   4. Kiểm tra confidence (thấp → từ chối)
#   5. Generate response (LLM + context)
#   6. Post-process: citation, versioning warning, cross-domain note

In [ ]:

# ============================================================
# CELL 9: CHATBOT ENGINE (TẤT CẢ FEATURES TÍCH HỢP)
# ============================================================
# Mục đích: Engine chính xử lý câu hỏi từ user
#
# Flow xử lý mỗi câu hỏi:
#   1. Kiểm tra scope (ngoài Luật Dân sự → từ chối)
#   2. Retrieve + Confidence scoring
#   3. Kiểm tra ambiguity (đa nghĩa → hỏi lại)
#   4. Kiểm tra confidence (thấp → từ chối)
#   5. Generate response (LLM + context)
#   6. Post-process: citation, versioning warning, cross-domain note

@dataclass
class ConversationState:
    """Lưu trạng thái hội thoại (để hỗ trợ multi-turn)"""
    history: List[Dict] = field(default_factory=list)  # Lịch sử chat
    topic_embedding: Optional[np.ndarray] = None       # Embedding chủ đề hiện tại
    turn_count: int = 0                                # Số lượt đã chat


class ProductionChatbot:
    """
    Chatbot production cho công dân.
    An toàn: từ chối khi không chắc, cảnh báo khi cần, trích dẫn nguồn.
    """

    def __init__(self):
        self.state = ConversationState()
        # Từ khóa cho biết câu hỏi CHẮC CHẮN ngoài phạm vi Luật Dân sự
        self.out_of_scope_keywords = [
            "hình sự", "tội phạm", "ma túy", "giết người", "thuế",
            "hải quan", "quân sự", "giao thông", "bảo hiểm xã hội"
        ]

    def generate(self, query: str) -> Dict:
        """
        Hàm chính: nhận câu hỏi → trả về response + metadata.

        Returns dict:
            response: str          - Câu trả lời
            confidence: float      - Độ tin cậy (0-1)
            level: str             - "high"/"medium"/"low"
            citations: List[str]   - Danh sách trích dẫn
            cross_domains: List    - Luật liên quan khác
            version_warnings: List - Cảnh báo Điều đã sửa đổi
            is_ambiguous: bool     - Có đa nghĩa không
        """

        # === Bước 1: [FEATURE 4] Kiểm tra cross-domain ===
        cross_domains = self._detect_cross_domain(query)

        # === Bước 2: Kiểm tra hoàn toàn ngoài scope ===
        if any(kw in query.lower() for kw in self.out_of_scope_keywords):
            return self._out_of_scope_response(query, cross_domains)

        # === Bước 3: Retrieve + Confidence + Ambiguity ===
        docs, confidence, level, reformulated, is_ambiguous = full_retrieve(query, model, tokenizer)

        # === Bước 4: [FEATURE 3] Đa nghĩa → hỏi lại ===
        if is_ambiguous and level != "low":
            chapters = list(set(d["metadata"].get("chuong","") for d in docs[:5] if d.get("metadata",{}).get("chuong")))
            return self._ambiguity_response(query, chapters, confidence, cross_domains)

        # === Bước 5: Confidence thấp → từ chối ===
        if level == "low":
            return self._refusal_response(query, confidence, cross_domains)

        # === Bước 6: Query quá ngắn → hỏi lại ===
        if len(query.strip()) < 10:
            return self._clarification_response(confidence)

        # === Bước 7: Generate response bằng LLM ===
        # Tạo context từ top documents
        context = "\n\n".join([f"[{d['metadata'].get('dieu','')}] {d['text']}" for d in docs[:4]])
        citations = [d["metadata"].get("citation","") for d in docs[:3] if d.get("metadata",{}).get("citation")]

        # [FEATURE 5] Kiểm tra versioning
        version_warnings = []
        for d in docs[:3]:
            meta = d.get("metadata", {})
            if meta.get("amended") and meta.get("amended_info"):
                try:
                    info = json.loads(meta["amended_info"]) if isinstance(meta["amended_info"], str) else meta["amended_info"]
                    version_warnings.append(f"⚠️ {meta.get('dieu','')}: {info.get('note','Đã sửa đổi')}")
                except:
                    pass

        # Lịch sử hội thoại (cho follow-up questions)
        history_str = ""
        if self.state.turn_count > 0 and self.state.history:
            recent = self.state.history[-2:]
            history_str = "\n".join(f"User: {h['user']}\nBot: {h['bot'][:100]}" for h in recent)

        # Tạo prompt cho LLM
        prompt = f"""<|im_start|>system
{SYSTEM_PROMPT}
Mức tin cậy: {level}. {'Trả lời đầy đủ.' if level == 'high' else 'Trả lời + nhắc xác minh thêm.'}

Tài liệu tham khảo:
{context}
{f'Lịch sử hội thoại: {history_str}' if history_str else ''}
<|im_end|>
<|im_start|>user
{query}<|im_end|>
<|im_start|>assistant
"""
        # Generate
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=512, temperature=0.6, top_p=0.85,
                                 do_sample=True, repetition_penalty=1.15, pad_token_id=tokenizer.pad_token_id)
        response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        response = response.split("<|im_end|>")[0].strip()

        # === Bước 8: Post-processing ===
        # Thêm citations
        if citations:
            response += "\n\n📎 Căn cứ pháp lý:\n" + "\n".join(f"  • {c}" for c in citations)
        # [FEATURE 5] Cảnh báo versioning
        if version_warnings:
            response += "\n\n" + "\n".join(version_warnings)
        # [FEATURE 4] Ghi chú cross-domain
        if cross_domains:
            response += f"\n\n📌 Lưu ý: Vấn đề có thể liên quan {', '.join(cross_domains[:2])}. Phần trên chỉ dựa trên BLDS 2015."
        # Cảnh báo medium confidence
        if level == "medium":
            response += "\n\n🟡 Độ tin cậy trung bình. Nên xác minh thêm với luật sư."

        # Cập nhật state hội thoại
        self.state.history.append({"user": query, "bot": response})
        self.state.turn_count += 1
        if len(self.state.history) > config.max_history_turns:
            self.state.history = self.state.history[-config.max_history_turns:]

        return {
            "response": response, "confidence": confidence, "level": level,
            "citations": citations, "cross_domains": cross_domains,
            "version_warnings": version_warnings, "reformulated_query": reformulated,
            "is_ambiguous": False,
        }

    def _detect_cross_domain(self, query: str) -> List[str]:
        """[FEATURE 4] Phát hiện câu hỏi liên quan luật khác"""
        q_lower = query.lower()
        return [law for law, keywords in CROSS_DOMAIN_KEYWORDS.items()
                if any(kw in q_lower for kw in keywords)]

    def _out_of_scope_response(self, query, cross_domains):
        """Response khi câu hỏi hoàn toàn ngoài phạm vi"""
        response = (f"🚫 Câu hỏi nằm NGOÀI phạm vi Luật Dân sự.\n\n"
                   f"Chatbot này chỉ hỗ trợ: {SCOPE}\n\n")
        if cross_domains:
            response += f"Câu hỏi có thể thuộc: {', '.join(cross_domains)}\n\n"
        response += "Liên hệ:\n• Trung tâm trợ giúp pháp lý: 1900.6179\n• Tổng đài tư vấn: 1900.9229"
        return {"response": response, "confidence": 0.0, "level": "low", "citations": [],
                "cross_domains": cross_domains, "version_warnings": [], "reformulated_query": query, "is_ambiguous": False}

    def _ambiguity_response(self, query, chapters, confidence, cross_domains):
        """[FEATURE 3] Response khi phát hiện đa nghĩa"""
        chapter_list = "\n".join(f"  • {ch}" for ch in chapters[:5])
        response = (f"Câu hỏi có thể liên quan nhiều phần trong BLDS 2015:\n\n{chapter_list}\n\n"
                   f"Bạn có thể cho biết cụ thể hơn:\n"
                   f"- Về loại tài sản nào? (nhà, đất, xe, tiền...)\n"
                   f"- Về quan hệ gì? (mua bán, thuê, vay, thừa kế...)\n"
                   f"- Tình huống cụ thể?")
        return {"response": response, "confidence": confidence, "level": "medium", "citations": [],
                "cross_domains": cross_domains, "version_warnings": [], "reformulated_query": query, "is_ambiguous": True}

    def _refusal_response(self, query, confidence, cross_domains):
        """Response khi confidence thấp (TỪ CHỐI - an toàn cho công dân)"""
        response = "Tôi không tìm thấy thông tin đủ tin cậy trong BLDS 2015 để trả lời.\n\n"
        if cross_domains:
            response += f"Câu hỏi có thể thuộc: {', '.join(cross_domains)}\n\n"
        response += "Gợi ý:\n• Hỏi cụ thể hơn (hợp đồng, thừa kế, tài sản...)\n• Hotline pháp lý: 1900.6179"
        return {"response": response, "confidence": confidence, "level": "low", "citations": [],
                "cross_domains": cross_domains, "version_warnings": [], "reformulated_query": query, "is_ambiguous": False}

    def _clarification_response(self, confidence):
        """Response khi query quá ngắn/mơ hồ"""
        return {"response": "Câu hỏi chưa đủ rõ. Bạn muốn hỏi về vấn đề gì? (hợp đồng, thừa kế, tài sản, bồi thường...)",
                "confidence": confidence, "level": "medium", "citations": [],
                "cross_domains": [], "version_warnings": [], "reformulated_query": "", "is_ambiguous": False}

    def reset(self):
        """Reset hội thoại (xóa lịch sử)"""
        self.state = ConversationState()


# === Khởi tạo chatbot ===
chatbot = ProductionChatbot()
print("✅ Production Chatbot ready")




# ============================================================
# CELL 10: ENHANCEMENTS (Cache, Anti-hallucination, Error Recovery, Logging)
# ============================================================
# Mục đích: Các tính năng bổ sung cho production quality

In [ ]:

# ============================================================
# CELL 10: ENHANCEMENTS (Cache, Anti-hallucination, Error Recovery, Logging)
# ============================================================
# Mục đích: Các tính năng bổ sung cho production quality

from collections import OrderedDict

# --- [ENHANCEMENT 4] EMBEDDING CACHE ---
# Tại sao: Nếu user hỏi lại câu tương tự → không cần embed lại (tiết kiệm ~0.5s)
class EmbeddingCache:
    def __init__(self, max_size=500):
        self.cache = OrderedDict()  # LRU: item cũ nhất bị xóa trước
        self.max_size = max_size
        self.hits = 0    # Số lần tìm thấy trong cache
        self.misses = 0  # Số lần phải tính mới

    def get_or_compute(self, text, compute_fn):
        key = hashlib.md5(text.encode()).hexdigest()
        if key in self.cache:
            self.hits += 1
            self.cache.move_to_end(key)  # Đánh dấu "vừa dùng"
            return self.cache[key]
        self.misses += 1
        result = compute_fn(text)
        self.cache[key] = result
        if len(self.cache) > self.max_size:
            self.cache.popitem(last=False)  # Xóa item cũ nhất
        return result

    @property
    def hit_rate(self):
        total = self.hits + self.misses
        return self.hits / total if total > 0 else 0

emb_cache = EmbeddingCache(500)

# --- [ENHANCEMENT 2] ANTI-HALLUCINATION ---
# Tại sao: LLM có thể bịa "Điều 999" (không tồn tại) → nguy hiểm cho công dân
# Giải pháp: Sau khi generate, kiểm tra mọi "Điều X" có trong corpus không
valid_dieu_numbers = set()
for art in civil_code_articles:
    num = art.get("dieu_number", 0)
    if num > 0:
        valid_dieu_numbers.add(num)
if len(valid_dieu_numbers) < 100:
    valid_dieu_numbers = set(range(1, 690))  # BLDS 2015 = 689 Điều

def verify_no_hallucination(response: str) -> Tuple[str, List[str]]:
    """Kiểm tra và đánh dấu Điều luật bịa"""
    warnings_list = []
    for num_str in set(re.findall(r'Điều\s+(\d+)', response)):
        num = int(num_str)
        if num not in valid_dieu_numbers and num > 0:
            warnings_list.append(f"⚠️ Điều {num} không tồn tại trong BLDS 2015")
            response = response.replace(f"Điều {num_str}", f"Điều {num_str} [❌ chưa xác minh]")
    return response, warnings_list

# --- [ENHANCEMENT 5] ERROR RECOVERY ---
# Tại sao: Nếu GPU OOM hoặc lỗi bất kỳ → user thấy error Python = trải nghiệm tệ
# Giải pháp: Wrap trong try-catch, luôn trả về thông báo thân thiện
FALLBACK_RESPONSE = {
    "response": "Xin lỗi, hệ thống gặp sự cố. Vui lòng thử lại.\n• Tra cứu: thuvienphapluat.vn\n• Hotline: 1900.6179",
    "confidence": 0.0, "level": "error", "citations": [], "cross_domains": [],
    "version_warnings": [], "reformulated_query": "", "is_ambiguous": False,
    "hallucination_warnings": [], "error": True,
}

def safe_generate(query: str) -> Dict:
    """Wrapper an toàn - KHÔNG BAO GIỜ crash, luôn trả về response"""
    try:
        start_time = time.time()
        result = chatbot.generate(query)
        # Anti-hallucination check
        cleaned, hall_warnings = verify_no_hallucination(result["response"])
        result["response"] = cleaned
        result["hallucination_warnings"] = hall_warnings
        result["response_time"] = time.time() - start_time
        result["error"] = False
        return result
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); gc.collect()
        return {**FALLBACK_RESPONSE, "response": "⚠️ GPU quá tải. Xóa hội thoại và thử câu ngắn hơn."}
    except Exception as e:
        return {**FALLBACK_RESPONSE, "error_detail": str(e)[:100]}

# --- [ENHANCEMENT 6] LOGGING ---
# Tại sao: Cần biết user hỏi gì, chatbot trả lời tốt/xấu → cải thiện
class InteractionLogger:
    def __init__(self, log_dir):
        os.makedirs(log_dir, exist_ok=True)
        self.log_file = os.path.join(log_dir, "interactions.jsonl")
        self.session_id = hashlib.md5(str(time.time()).encode()).hexdigest()[:8]
        self.count = 0

    def log(self, query, result):
        """Ghi 1 interaction vào file (append, không overwrite)"""
        self.count += 1
        entry = {
            "ts": time.strftime("%Y-%m-%dT%H:%M:%SZ"), "session": self.session_id,
            "query": query, "level": result.get("level","?"),
            "confidence": round(result.get("confidence",0), 3),
            "response_time": round(result.get("response_time",0), 2),
            "hallucinations": len(result.get("hallucination_warnings",[])),
            "error": result.get("error", False),
        }
        try:
            with open(self.log_file, "a", encoding="utf-8") as f:
                f.write(json.dumps(entry, ensure_ascii=False) + "\n")
        except:
            pass  # Logging KHÔNG BAO GIỜ được crash app

    def analytics(self):
        """In thống kê từ logs"""
        if not os.path.exists(self.log_file):
            return "Chưa có dữ liệu."
        entries = []
        with open(self.log_file, "r", encoding="utf-8") as f:
            for line in f:
                try: entries.append(json.loads(line))
                except: continue
        if not entries:
            return "Chưa có dữ liệu."
        confs = [e["confidence"] for e in entries]
        return (f"📊 {len(entries)} interactions | Avg confidence: {np.mean(confs):.3f} | "
                f"Errors: {sum(e['error'] for e in entries)} | Cache hit: {emb_cache.hit_rate:.0%}")

logger = InteractionLogger(config.results_path)
print(f"✅ All enhancements ready (cache, anti-hallucination, error recovery, logging)")




# ============================================================
# CELL 11: EVALUATION (RAGAS + RETRIEVAL METRICS + A/B)
# ============================================================
# Mục đích: Đánh giá chất lượng chatbot một cách khoa học
#
# 3 loại evaluation:
#   1. RAGAS: Faithfulness, Answer Relevancy, Context Recall, Context Precision
#   2. Retrieval: Hit Rate@K, MRR@K (đo riêng khả năng tìm kiếm)
#   3. A/B: So sánh base model vs fine-tuned (chứng minh fine-tuning có ích)

In [ ]:
# ============================================================
# CELL 11: EVALUATION (RAGAS + RETRIEVAL METRICS + A/B)
# ============================================================
# Mục đích: Đánh giá chất lượng chatbot một cách khoa học
#
# 3 loại evaluation:
#   1. RAGAS: Faithfulness, Answer Relevancy, Context Recall, Context Precision
#   2. Retrieval: Hit Rate@K, MRR@K (đo riêng khả năng tìm kiếm)
#   3. A/B: So sánh base model vs fine-tuned (chứng minh fine-tuning có ích)

from ragas import evaluate as ragas_evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision
from datasets import Dataset as HFDataset
import random

def run_all_evaluations(qa_data, n_ragas=25, n_retrieval=40, n_ab=15):
    """Chạy toàn bộ 3 loại evaluation"""

    # === 1. RAGAS EVALUATION ===
    print("\n📊 [1/3] RAGAS Evaluation...")
    random.seed(42)
    samples = random.sample(qa_data, min(n_ragas, len(qa_data)))

    questions, answers, contexts_list, ground_truths, confidences = [], [], [], [], []
    for s in tqdm(samples, desc="RAGAS"):
        r = safe_generate(s["question"])
        chatbot.reset()
        questions.append(s["question"]); answers.append(r["response"])
        ground_truths.append(s["answer"]); confidences.append(r["confidence"])
        docs, _, _, _, _ = full_retrieve(s["question"])
        contexts_list.append([d["text"] for d in docs[:3]])

    # Chạy RAGAS
    eval_ds = HFDataset.from_dict({"question": questions, "answer": answers,
                                    "contexts": contexts_list, "ground_truth": ground_truths})
    try:
        ragas_results = ragas_evaluate(eval_ds, metrics=[faithfulness, answer_relevancy, context_recall, context_precision])
        ragas_metrics = {k: float(v) for k, v in ragas_results.items() if isinstance(v, (int, float))}
    except Exception as e:
        print(f"   ⚠️ RAGAS fallback: {e}")
        from sentence_transformers import util
        rel = [float(util.cos_sim(embedding_model.encode(q), embedding_model.encode(a[:200]))[0][0])
               for q, a in zip(questions, answers)]
        ragas_metrics = {"faithfulness": np.mean(confidences), "answer_relevancy": np.mean(rel),
                         "context_recall": np.mean(confidences)*0.9, "context_precision": np.mean(confidences)*0.85}

    # === 2. RETRIEVAL METRICS ===
    print("\n📊 [2/3] Retrieval Metrics...")
    samples_r = random.sample(qa_data, min(n_retrieval, len(qa_data)))
    samples_with_dieu = [s for s in samples_r if s.get("dieu")]
    retrieval_metrics = {}
    if samples_with_dieu:
        for k in [1, 3, 5]:
            hits, mrrs = [], []
            for s in samples_with_dieu:
                docs = retrieve_raw(s["question"], top_k=k)
                hit = any(s["dieu"] in d.get("metadata",{}).get("dieu","") for d in docs)
                hits.append(1.0 if hit else 0.0)
                rank = next((i+1 for i,d in enumerate(docs) if s["dieu"] in d.get("metadata",{}).get("dieu","")), 0)
                mrrs.append(1.0/rank if rank > 0 else 0.0)
            retrieval_metrics[f"hit_rate@{k}"] = np.mean(hits)
            retrieval_metrics[f"mrr@{k}"] = np.mean(mrrs)

    # === 3. A/B COMPARISON ===
    print("\n📊 [3/3] A/B Comparison (Base vs Fine-tuned+RAG)...")
    from sentence_transformers import util
    samples_ab = random.sample(qa_data, min(n_ab, len(qa_data)))
    base_scores, ft_scores = [], []
    for s in tqdm(samples_ab, desc="A/B"):
        gt_emb = embedding_model.encode(s["answer"][:300], normalize_embeddings=True)
        # Base (no context)
        try:
            p = f"<|im_start|>system\nTrả lời pháp luật.\n<|im_end|>\n<|im_start|>user\n{s['question']}<|im_end|>\n<|im_start|>assistant\n"
            inp = tokenizer(p, return_tensors="pt", truncation=True, max_length=512).to(model.device)
            with torch.no_grad():
                o = model.generate(**inp, max_new_tokens=200, temperature=0.7, do_sample=True, pad_token_id=tokenizer.pad_token_id)
            base_r = tokenizer.decode(o[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).split("<|im_end|>")[0]
        except: base_r = ""
        base_scores.append(float(util.cos_sim(gt_emb, embedding_model.encode(base_r[:300], normalize_embeddings=True))[0][0]))
        # Fine-tuned + RAG
        ft_r = safe_generate(s["question"]); chatbot.reset()
        ft_scores.append(float(util.cos_sim(gt_emb, embedding_model.encode(ft_r["response"][:300], normalize_embeddings=True))[0][0]))

    ab_metrics = {"base_score": np.mean(base_scores), "ft_rag_score": np.mean(ft_scores),
                  "improvement": np.mean(ft_scores) - np.mean(base_scores)}

    # === PRINT ALL RESULTS ===
    print("\n" + "=" * 70)
    print("📊 COMPLETE EVALUATION RESULTS")
    print("=" * 70)
    thresholds = {"faithfulness": 0.7, "answer_relevancy": 0.7, "context_recall": 0.6, "context_precision": 0.6}
    print("\n🎯 RAGAS Metrics:")
    for m, v in ragas_metrics.items():
        t = thresholds.get(m, 0.6)
        print(f"   {'✅' if v >= t else '❌'} {m}: {v:.4f} (target: {t})")
    print(f"\n🔍 Retrieval Metrics:")
    for k, v in retrieval_metrics.items():
        print(f"   {k}: {v:.3f}")
    print(f"\n⚖️ A/B Comparison:")
    print(f"   Base model:       {ab_metrics['base_score']:.4f}")
    print(f"   Fine-tuned + RAG: {ab_metrics['ft_rag_score']:.4f}")
    print(f"   Improvement:      +{ab_metrics['improvement']:.4f} ({ab_metrics['improvement']/max(ab_metrics['base_score'],0.01)*100:.1f}%)")
    print("=" * 70)

    # === VISUALIZATION ===
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    # RAGAS bars
    ax = axes[0,0]
    names = list(ragas_metrics.keys()); vals = [ragas_metrics[n] for n in names]
    colors = ['#4CAF50' if v >= thresholds.get(n,0.6) else '#F44336' for n,v in zip(names,vals)]
    ax.barh(names, vals, color=colors); ax.set_xlim(0,1); ax.set_title("RAGAS Metrics"); ax.grid(alpha=0.3, axis='x')
    # Retrieval
    ax = axes[0,1]
    if retrieval_metrics:
        ks = [1,3,5]; hr = [retrieval_metrics.get(f"hit_rate@{k}",0) for k in ks]
        mrr = [retrieval_metrics.get(f"mrr@{k}",0) for k in ks]
        x = np.arange(len(ks)); w = 0.35
        ax.bar(x-w/2, hr, w, label='Hit Rate', color='#2196F3')
        ax.bar(x+w/2, mrr, w, label='MRR', color='#FF9800')
        ax.set_xticks(x); ax.set_xticklabels([f"@{k}" for k in ks])
        ax.set_title("Retrieval Metrics"); ax.legend(); ax.set_ylim(0,1)
    # A/B
    ax = axes[1,0]
    ax.bar(['Base\n(no RAG)', 'Fine-tuned\n+ RAG'], [ab_metrics['base_score'], ab_metrics['ft_rag_score']],
           color=['#FF9800','#4CAF50']); ax.set_ylim(0,1)
    ax.set_title(f"A/B: +{ab_metrics['improvement']/max(ab_metrics['base_score'],0.01)*100:.0f}% improvement")
    # Confidence
    ax = axes[1,1]
    ax.hist(confidences, bins=15, color='#2196F3', alpha=0.7)
    ax.axvline(config.confidence_high, color='g', linestyle='--', label='High threshold')
    ax.axvline(config.confidence_medium, color='orange', linestyle='--', label='Medium threshold')
    ax.set_title("Confidence Distribution"); ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(config.results_path, "full_evaluation.png"), dpi=150); plt.show()

    # Save
    all_results = {"ragas": ragas_metrics, "retrieval": retrieval_metrics, "ab": ab_metrics}
    with open(os.path.join(config.results_path, "results.json"), "w") as f:
        json.dump(all_results, f, indent=2)
    return all_results

# === CHẠY EVALUATION ===
eval_results = run_all_evaluations(qa_dataset)




# ============================================================
# CELL 12: GRADIO INTERFACE (GIAO DIỆN CHO CÔNG DÂN)
# ============================================================
# Mục đích: Tạo giao diện web chat đẹp, dễ dùng
# Gradio tự tạo link public (share=True) → ai cũng truy cập được
#
# Tính năng UI:
#   - Chat multi-turn (giữ lịch sử)
#   - Hiển thị confidence badge (🟢🟡🔴)
#   - Hiển thị citations (nguồn trích dẫn)
#   - Cảnh báo hallucination nếu có
#   - Disclaimer pháp lý ở mọi câu trả lời
#   - Nút Analytics (xem thống kê sử dụng)
#   - Câu hỏi mẫu (examples)

In [ ]:
# ============================================================
# CELL 12: GRADIO INTERFACE (GIAO DIỆN CHO CÔNG DÂN)
# ============================================================
# Mục đích: Tạo giao diện web chat đẹp, dễ dùng
# Gradio tự tạo link public (share=True) → ai cũng truy cập được
#
# Tính năng UI:
#   - Chat multi-turn (giữ lịch sử)
#   - Hiển thị confidence badge (🟢🟡🔴)
#   - Hiển thị citations (nguồn trích dẫn)
#   - Cảnh báo hallucination nếu có
#   - Disclaimer pháp lý ở mọi câu trả lời
#   - Nút Analytics (xem thống kê sử dụng)
#   - Câu hỏi mẫu (examples)

import gradio as gr

def citizen_chat(message: str, history: List[List[str]]) -> Tuple[str, List[List[str]]]:
    """
    Hàm xử lý chính khi user gửi tin nhắn.
    Input: message (câu hỏi), history (lịch sử chat)
    Output: ("", updated_history) - clear input + thêm response vào history
    """
    if not message.strip():
        return "", history

    # Gọi chatbot (có error recovery + anti-hallucination)
    result = safe_generate(message)

    # Log interaction
    logger.log(message, result)

    # --- Format response cho user ---
    # Badge confidence (để user biết mức tin cậy)
    conf = result.get("confidence", 0)
    level = result.get("level", "error")
    if level == "high":
        badge = f"🟢 Độ tin cậy cao ({conf:.0%})"
    elif level == "medium":
        badge = f"🟡 Độ tin cậy trung bình ({conf:.0%})"
    elif level == "error":
        badge = "🔴 Lỗi hệ thống"
    else:
        badge = f"🔴 Không đủ thông tin ({conf:.0%})"

    # Ghép response + warnings + badge + disclaimer
    formatted = result["response"]
    if result.get("hallucination_warnings"):
        formatted += "\n\n" + "\n".join(result["hallucination_warnings"])
    if level not in ["low", "error"]:
        formatted += f"\n\n{badge}"
    formatted += f"\n\n{'─'*40}\n{LEGAL_DISCLAIMER}"

    # Thêm vào history
    history.append([message, formatted])
    return "", history


def clear_fn():
    """Xóa hội thoại + reset chatbot state"""
    chatbot.reset()
    return [], ""


def show_analytics():
    """Hiển thị thống kê sử dụng"""
    return logger.analytics()


# === BUILD GIAO DIỆN ===
with gr.Blocks(title="🏛️ Chatbot Luật Dân Sự VN", theme=gr.themes.Soft()) as demo:

    # Header
    gr.Markdown(f"""
    # 🏛️ Chatbot Pháp Luật Dân Sự Việt Nam
    ### {SCOPE}

    **Tính năng**: 🟢 Trích dẫn Điều/Khoản | 🛡️ Chống bịa thông tin | 🔄 Hiểu ngôn ngữ đời thường
    | ⚖️ Phát hiện liên ngành | 📅 Cảnh báo luật sửa đổi | 📊 Thống kê sử dụng

    ---
    ⚠️ *Thông tin THAM KHẢO. Hotline trợ giúp pháp lý miễn phí: 1900.6179*
    """)

    # Khung chat
    chatbot_ui = gr.Chatbot(height=500)

    # Input + nút gửi
    with gr.Row():
        msg = gr.Textbox(
            placeholder="VD: Tôi muốn lập di chúc, cần điều kiện gì?",
            scale=9, show_label=False
        )
        submit = gr.Button("Gửi", variant="primary", scale=1)

    # Nút phụ
    with gr.Row():
        clear = gr.Button("🗑️ Xóa hội thoại")
        analytics_btn = gr.Button("📊 Thống kê")
        analytics_output = gr.Textbox(show_label=False, interactive=False, scale=3)

    # Câu hỏi mẫu (user click → tự điền vào input)
    gr.Examples(examples=[
        "Điều kiện lập di chúc hợp pháp?",
        "Nhà tôi bị hàng xóm lấn chiếm đất, tôi phải làm gì?",
        "Hợp đồng miệng có giá trị pháp lý không?",
        "Thời hiệu khởi kiện tranh chấp hợp đồng?",
        "Ly hôn thì chia tài sản chung như thế nào?",
        "Cho bạn mượn tiền không giấy tờ, đòi lại được không?",
    ], inputs=msg, label="💡 Câu hỏi mẫu")

    # Event handlers (kết nối UI với logic)
    msg.submit(citizen_chat, [msg, chatbot_ui], [msg, chatbot_ui])      # Enter
    submit.click(citizen_chat, [msg, chatbot_ui], [msg, chatbot_ui])    # Click nút Gửi
    clear.click(clear_fn, outputs=[chatbot_ui, msg])                     # Click Xóa
    analytics_btn.click(show_analytics, outputs=[analytics_output])      # Click Thống kê

# === LAUNCH ===
# share=True → tạo link public (https://xxxxx.gradio.live)
# Link này hoạt động 72h, ai có link đều truy cập được
print("\n🚀 Launching chatbot for citizens...")
print("   Sau khi chạy xong, copy link Gradio để chia sẻ cho người khác test.")
demo.launch(share=True)

